<span style = "color: aqua; font-size: 36px;"># DELTA LAKE</span>

Local instance to manage MatrizActividades

## HORAS EXTRA

### PASO 1. Inicializacion y conexion con Deltalake

In [1]:
# Importaciones - Ubicacion de DeltaLake - conversion a Pandas
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl import load_workbook
from openpyxl.styles import Alignment
from openpyxl.styles import Alignment, numbers
from openpyxl.utils import get_column_letter

import shutil
import os
import sys
import time
from pathlib import Path

import pandas as pd
import numpy as np
import re
from natsort import order_by_index, index_natsorted
import logging
from deltalake import DeltaTable, write_deltalake
from datetime import datetime, time
from datetime import date as toDate
import eerssa.utils

logging.basicConfig(level=logging.INFO)

DELTA_TABLE_PATH_ON_HOST = "/home/vlad/delta_V30"

table_path = DELTA_TABLE_PATH_ON_HOST

# ------  Delta Lake  --------- #

if not DeltaTable.is_deltatable(table_path):
    print(
        f"No se ha encontrado la base de datos PARQUET-DELTALAKE en la direccion:\n NO_DELTA_LAKE : {table_path}" )
else:
    dt = DeltaTable(table_path)
    df = dt.to_pandas()
    #df.info()
    print(f"\n\nConectado a la tabla Delta Lake en: {table_path}")

# Hot Reload Functions
%load_ext autoreload
%autoreload 2


Success!!!


Conectado a la tabla Delta Lake en: /home/vlad/delta_V30


### <span style="color: red; font-size: 22px;">DEFINICIÓN DE FECHAS</span>

Fecha inicial y final para generar dataframes y reportes, Estas fechas controlan la longitud de los dataframe


In [8]:
reporte_inicia = toDate( 2026, 4, 1 )
reporte_finaliza = toDate( 2026,4, 30 )

### RELOAD: Recargar DeltaLake & Dataframe (df)

Reconstruir el Dataframe completo `df` desde DeltaLake.

> ☝️ Se puede optimizar consultando solamente las fechas deseadas. 

In [32]:
dt = DeltaTable(table_path)
df = dt.to_pandas()
print(f">>> VERSION Actual del Deltalake: {dt.version()}")


>>> VERSION Actual del Deltalake: 5319


### <span style="color: red; font-size: 22px;">**GENERACION DE NUEVOS ARCHIVOS EXCEL**</span> 

**Esto creará nuevos archivos con la Fecha Actual!**


In [4]:
# EJECUTAR: Creación del nuevo archivo desde el template

template_path = os.path.join('models','template.xlsx')
he_template_path = os.path.join('models','hora_extra_template.xlsx')

# Pickle con DB_HE_2026
db_he_path = os.path.join( 'models','DB_HE_2026.pkl' ) 

today =  datetime.today().strftime('%Y%m%d')
output_filename = f"actividades_{today}.xlsx"
output_he_file  = f"base_HE_{today}.xlsx"

output_path = os.path.join('reporte', output_filename)
output_he   = os.path.join('reporte', output_he_file )

shutil.copyfile(template_path, output_path)
print(f"Se ha copiado Excel para OT en {output_path}")


Se ha copiado Excel para OT en reporte/actividades_20260512.xlsx


## EXPORTAR Actividades a Excel

Creación del archivo Excel donde se guardara el Reporte:
- Es una copia del archivo: `template.xlsx`
- Se insertarán el dataframe: `filtered_df` definido más adelante. 

Modificación de la informacion en `df` para acoplarla y sea compatible con Excel

### Generar Dataframe todas las OT entre las fechas

In [27]:
# Genera un DataFrame entre las fechas para proceder a la exportacion
excel = df.copy()

# ── DataFrame conditioning ──────────────────────────────────────────────────────────
excel['Fecha'] = excel['Fecha'].apply(lambda x: eerssa.utils.soloFecha_SinTimezone( x ))
excel['Date']  = excel['Fecha'].apply(lambda x: eerssa.utils.toDateObject( x ))
excel['Duracion'] = excel['Duracion']/(60*24)

filtered_df = excel[ (excel['Date'] >= reporte_inicia ) & ( excel['Date'] <= reporte_finaliza )]
filtered_df = filtered_df.drop(columns=['Date'])
filtered_df.loc[filtered_df['Cuenta'] == 'se_labora', 'HorasExtra'] = 'No'
filtered_df.loc[( filtered_df['Item'] == 1) & (filtered_df['Cuenta'] == 'informativa'), 'HorasExtra'] = 'No'

filtered_df = filtered_df.iloc[index_natsorted(zip(filtered_df['Archivo'], filtered_df['Item']))]

print(f"Dataframe cargado con {len(filtered_df)} filas")

Dataframe cargado con 954 filas


### Generar Dataframe _solamente_ OTs con Horas Extra entre las Fechas

In [9]:
# Genera un DataFrame entre las fechas Y CON HORA EXTRA

excel = df.copy()

# ── DataFrame conditioning ──────────────────────────────────────────────────────────
excel['Fecha'] = excel['Fecha'].apply(lambda x: eerssa.utils.soloFecha_SinTimezone( x ))
excel['Date']  = excel['Fecha'].apply(lambda x: eerssa.utils.toDateObject( x ))
excel['Duracion'] = excel['Duracion']/(60*24)

filtered_df = excel[ (excel['Date'] >= reporte_inicia ) & ( excel['Date'] <= reporte_finaliza )]
filtered_df = filtered_df.drop(columns=['Date'])
filtered_df.loc[filtered_df['Cuenta'] == 'se_labora', 'HorasExtra'] = 'No'
filtered_df.loc[( filtered_df['Item'] == 1) & (filtered_df['Cuenta'] == 'informativa'), 'HorasExtra'] = 'No'

# Step 1: Get unique 'id_ot' values where 'HorasExtra' is 'Si'
# We use .loc to filter rows and select the 'id_ot' column, then .unique() to get distinct IDs
ids_with_extra = filtered_df.loc[filtered_df['HorasExtra'] == 'Si', 'id_ot'].unique()

# Step 2: Filter the dataframe to keep all rows where 'id_ot' is in our list
filtered_df = filtered_df[filtered_df['id_ot'].isin(ids_with_extra)]


filtered_df = filtered_df.iloc[index_natsorted(zip(filtered_df['Archivo'], filtered_df['Item']))]

print(f"Dataframe cargado con {len(filtered_df)} filas")

Dataframe cargado con 1405 filas


In [17]:
filtered_df

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
350629,1,informativa,Revisión de órdenes de trabajo de la zona\nOrg...,PROG,·,No,No,No,RUTINARIA,·,...,2026-04-24 07:30:00,2026-04-24 11:00:00,0.145833,PALACIOS MERINO ERNESTO VLADIMIR,0,No,4-36,"Zamora, Gualaquiza",175603,OT [00] 00_Jefe Zonal 2026-04-24 (004) EP.pdf
373845,3,transporte,Traslado de Zamora hasta El Pangui,TRANSP,·,No,No,No,TRANSPORTE,·,...,2026-04-24 11:00:00,2026-04-24 13:00:00,0.083333,PALACIOS MERINO ERNESTO VLADIMIR,0,No,4-36,"Zamora, Gualaquiza",175603,OT [00] 00_Jefe Zonal 2026-04-24 (004) EP.pdf
3557,4,informativa,"En Agencia El Pangui, reunion con Luis Agurto ...",PROG,·,No,No,No,PREVENTIVO,·,...,2026-04-24 13:00:00,2026-04-24 14:00:00,0.041667,PALACIOS MERINO ERNESTO VLADIMIR,0,No,4-36,"Zamora, Gualaquiza",175603,OT [00] 00_Jefe Zonal 2026-04-24 (004) EP.pdf
362225,5,REDES,Se coordinan maniobras con el Centro de Contro...,PROG,Bomboiza,No,No,No,PREVENTIVO,·,...,2026-04-24 14:00:00,2026-04-24 15:30:00,0.062500,PALACIOS MERINO ERNESTO VLADIMIR,0,No,4-36,"Zamora, Gualaquiza",175603,OT [00] 00_Jefe Zonal 2026-04-24 (004) EP.pdf
362226,7,lunch,LUNCH EN CHUCHUMBLETZA,ALIMEN,·,No,No,No,LUNCH,·,...,2026-04-24 15:30:00,2026-04-24 16:30:00,0.041667,PALACIOS MERINO ERNESTO VLADIMIR,0,No,4-36,"Zamora, Gualaquiza",175603,OT [00] 00_Jefe Zonal 2026-04-24 (004) EP.pdf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401032,16,transporte,Traslado Zapotillo a Ag Gualaquiza,TRANSP,·,No,No,No,TRANSPORTE,·,...,2026-04-24 16:40:00,2026-04-24 17:00:00,0.013889,CARTUCHE SILVA GERARDO PATRICIO,1,No,2-107,"Gualaquiza, Zapotillo Cuchipamba, Nueva Tarqui...",175420,OT [24] Agencia Gualaquiza 2026-04-24 (062) GC...
362224,17,transporte,Traslado Ag Gualaquiza a Bomboiza,TRANSP,·,No,No,No,TRANSPORTE,·,...,2026-04-24 17:45:00,2026-04-24 18:15:00,0.020833,"GC, MA",1,Si,2-107,"Gualaquiza, Zapotillo Cuchipamba, Nueva Tarqui...",175420,OT [24] Agencia Gualaquiza 2026-04-24 (062) GC...
350627,18,ACOMETIDAS,"Evento reportado por Centro de Control, Sr. Br...",NO PROG,Bomboiza,No,No,No,CORRECTIVO,·,...,2026-04-24 18:15:00,2026-04-24 18:55:00,0.027778,"GC, MA",1,Si,2-107,"Gualaquiza, Zapotillo Cuchipamba, Nueva Tarqui...",175420,OT [24] Agencia Gualaquiza 2026-04-24 (062) GC...
404949,19,transporte,Traslado de Bomboiza a Ag Gualaquiza,TRANSP,·,No,No,No,TRANSPORTE,·,...,2026-04-24 18:55:00,2026-04-24 19:24:00,0.020139,"GC, MA",1,Si,2-107,"Gualaquiza, Zapotillo Cuchipamba, Nueva Tarqui...",175420,OT [24] Agencia Gualaquiza 2026-04-24 (062) GC...


In [19]:
pd.to_datetime("2026-04-30").date()

datetime.date(2026, 4, 30)

### EXPORTAR Dataframe a Excel
Se exporta al archivo: 'reporte/actividades_{today}.xlsx' ya sea todas las OT o solamente aquellas que tienen Horas Extra


In [29]:
# Genera el archivo de Excel desde el `filtered_df`

# Load the copied workbook
wb = load_workbook(output_path)

# Access the second sheet (0-based index; change if needed)
sheet = wb.worksheets[1]  # Or wb['Sheet2'] if named

# Optional: Clear existing data from row 2 down (preserves headers and formats)
for row in sheet.iter_rows(min_row=2, max_row=sheet.max_row, min_col=1, max_col=sheet.max_column):
    for cell in row:
        cell.value = None

# Insert DataFrame starting from row 2 (skip headers in DF)
for r_idx, row in enumerate(dataframe_to_rows(filtered_df, index=False, header=False), 2):
    for c_idx, value in enumerate(row, 1):
        sheet.cell(row=r_idx, column=c_idx, value=value)

# Load validation rules
max_row = sheet.max_row

validate_list = '=LISTAS!B$3:B$30' # Cuenta
dataValidation = DataValidation( type="list", formula1=validate_list, allow_blank=True )
dataValidation.add( f'B2:B{max_row}' )
sheet.add_data_validation(dataValidation)

validate_list = '=LISTAS!E$3:E$30' #Tipo
dataValidation = DataValidation( type="list", formula1=validate_list, allow_blank=True )
dataValidation.add( f'I2:I{max_row}' )
sheet.add_data_validation(dataValidation)

validate_list = '=LISTAS!G$3:G$30' # Actividad
dataValidation = DataValidation( type="list", formula1=validate_list, allow_blank=True )
dataValidation.add( f'D2:D{max_row}' )
sheet.add_data_validation(dataValidation)

validate_list = '=LISTAS!I$3:I$100' # Alimentador
dataValidation = DataValidation( type="list", formula1=validate_list, allow_blank=True )
dataValidation.add( f'E2:E{max_row}' )
sheet.add_data_validation(dataValidation)

validate_list = '"Si,No"' # Binario 
dataValidation = DataValidation( type="list", 
                                formula1=validate_list, 
                                allow_blank=False, 
                                showErrorMessage=True )
dataValidation.add( f'F2:F{max_row}' ) # Primario
dataValidation.add( f'G2:G{max_row}' ) # Desconexion
dataValidation.add( f'H2:H{max_row}' ) # SIG
dataValidation.add( f'S2:S{max_row}' ) # HorasExtra
sheet.add_data_validation(dataValidation)


# Save the modified workbook
wb.save(output_path)
print(f"✅ Se ha generado el archivo de EXCEL en -> {output_path}")


✅ Se ha generado el archivo de EXCEL en -> reporte/actividades_20260512.xlsx


/home/vlad/GIT/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


## 🐞 Recuperación de ACTIVIDADES desde Excel
Hay algun error cuando se recupera Todas las Actividades Abril 2026

<span style="color: red; font-size: 22px;">**NOMBRE DEL ARCHIVO** para cargar de Excel a Pandas</span>

In [31]:
excel_file_path = output_path
# Aqui puedo escoger recargar desde algun archivo Excel para pruebas,
# por defecto es el generado en el paso anterior

In [97]:
# 1. Load the modified Excel file
modified_df = pd.read_excel(excel_file_path, sheet_name = "ACTIVIDADES")
modified_df = modified_df.dropna(subset=['Cuadrilla'])

# 2. Se vuelva a colocar el String de TimeZone en la Fecha
modified_df['Fecha'] = modified_df['Fecha'].apply(lambda x: eerssa.utils.ColocarTimezone( x ))


#4. Convert everything to datetime objects first, then format them all as uniform strings
modified_df['InicioEvento'] = pd.to_datetime(modified_df['InicioEvento'], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')
modified_df['FinEvento'] = pd.to_datetime(modified_df['FinEvento'], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')


# 4. Ensure Schema Consistency
# Excel often introduces new columns (like empty comments) or reorders them.
# We force the modified_df to have the same columns as the original df.
modified_df = modified_df[df.columns]



In [391]:
start_times = pd.to_datetime(modified_df['Ini'])
end_times = pd.to_datetime(modified_df['Fin'])
time_difference = end_times - start_times
time_difference

0      0 days 01:00:00
1      0 days 01:15:00
2      0 days 02:45:00
3      0 days 01:00:00
4      0 days 03:50:00
             ...      
1993   0 days 02:20:00
1994   0 days 00:40:00
1995   0 days 00:45:00
1996   0 days 00:25:00
1997   0 days 03:30:00
Length: 1998, dtype: timedelta64[ns]

In [99]:
modified_df

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
0,1,informativa,En la Agencia Zamora se coordina trabajo conju...,PROG,·,No,No,No,RUTINARIA,·,...,2026-04-01 08:00:00,2026-04-01 09:00:00,60,RIOS RIOS FRANCISCO FERNANDO,0,No,2-110,"Zamora, Loja",173886,OT [01] Cuadrilla Zamora 2026-04-01 (006) FR.pdf
1,2,transporte,Desde la Agencia Zamora me traslado hacia la C...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2026-04-01 09:00:00,2026-04-01 10:15:00,75,RIOS RIOS FRANCISCO FERNANDO,0,No,2-110,"Zamora, Loja",173886,OT [01] Cuadrilla Zamora 2026-04-01 (006) FR.pdf
2,3,informativa,"En los Talleres de la EERSSA, se retira filtro...",PROG,·,No,No,No,PREDICTIVO,·,...,2026-04-01 10:15:00,2026-04-01 13:00:00,165,RIOS RIOS FRANCISCO FERNANDO,0,No,2-110,"Zamora, Loja",173886,OT [01] Cuadrilla Zamora 2026-04-01 (006) FR.pdf
3,4,lunch,LUNCH EN LOJA.,ALIMEN,·,No,No,No,LUNCH,·,...,2026-04-01 13:00:00,2026-04-01 14:00:00,60,RIOS RIOS FRANCISCO FERNANDO,0,No,2-110,"Zamora, Loja",173886,OT [01] Cuadrilla Zamora 2026-04-01 (006) FR.pdf
4,5,informativa,"Loja, Edificio Central de la EERSSA, se contin...",PROG,·,No,No,No,PREDICTIVO,·,...,2026-04-01 14:00:00,2026-04-01 17:50:00,230,RIOS RIOS FRANCISCO FERNANDO,0,No,2-110,"Zamora, Loja",173886,OT [01] Cuadrilla Zamora 2026-04-01 (006) FR.pdf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
898,5,se_labora,Se labora: CB y GC de 16:16 a 18:24,LABORA,·,No,No,No,·,·,...,2026-04-04 00:00:01,2026-04-04 00:00:02,0,BUELE UYAGUARI CESAR CRISTIAN,1,No,2-107,Gualaquiza,174083,OT [24] Agencia Gualaquiza 2026-04-04 (061) CB...
899,1,transporte,"Gualaquiza, recepcion reclamo Centro Control, ...",TRANSP,·,No,No,No,TRANSPORTE,·,...,2026-04-05 08:48:00,2026-04-05 09:20:00,32,"CB, GC",1,Si,2-107,Gualaquiza,174084,OT [24] Agencia Gualaquiza 2026-04-05 (061) CB...
900,2,REDES,"Chumpias, revision LMT estructura #028535 a #0...",NO PROG,Bomboiza,No,No,No,CORRECTIVO,·,...,2026-04-05 09:20:00,2026-04-05 10:25:00,65,"CB, GC",1,Si,2-107,Gualaquiza,174084,OT [24] Agencia Gualaquiza 2026-04-05 (061) CB...
901,3,transporte,servicio.\nTraslado Gualaquiza.,TRANSP,·,No,No,No,TRANSPORTE,·,...,2026-04-05 10:25:00,2026-04-05 10:49:00,24,"CB, GC",1,Si,2-107,Gualaquiza,174084,OT [24] Agencia Gualaquiza 2026-04-05 (061) CB...


In [32]:
excel_file_path

'reporte/actividades_20260512.xlsx'

In [33]:
# 1. Load the modified Excel file
modified_df = pd.read_excel(excel_file_path, sheet_name = "ACTIVIDADES")

# 2. Se vuelva a colocar el String de TimeZone en la Fecha
modified_df['Fecha'] = modified_df['Fecha'].apply(lambda x: eerssa.utils.ColocarTimezone( x ))

# 3. Convertir de String a TimeObject y se vuelve a calcular la duración en minutos
modified_df['Ini'] = pd.to_datetime(modified_df['InicioEvento'], errors='coerce')
modified_df['Fin'] = pd.to_datetime(modified_df['FinEvento'], errors='coerce')

modified_df['Duracion'] = eerssa.utils.calcular_minutos_transcurridos(
    modified_df['Ini'],
    modified_df['Fin']
)

modified_df = modified_df.drop(columns=['Ini', 'Fin'])

#4. Convert everything to datetime objects first, then format them all as uniform strings
modified_df['InicioEvento'] = pd.to_datetime(modified_df['InicioEvento'], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')
modified_df['FinEvento'] = pd.to_datetime(modified_df['FinEvento'], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')


# 4. Ensure Schema Consistency
# Excel often introduces new columns (like empty comments) or reorders them.
# We force the modified_df to have the same columns as the original df.
modified_df = modified_df[df.columns]


### Actulizar en DeltaLake

In [46]:
modified_df = df

In [ ]:
# Con el archivo modificado en Excel, se actualizan las filas en DeltaLake
try:
    # --- Start of new logic ---
    # 1. Get a list of all unique 'id_ot' values from the source DataFrame.
    ids_to_update = modified_df['id_ot'].unique()

    # 2. Format the list into a SQL-compatible string like "(101, 102, 103)".
    # This is crucial for the IN clause to work correctly.
    ids_predicate_string = ", ".join(map(str, ids_to_update))
    
    # 3. Define the delete predicate to scope deletions to only the OTs being updated.
    delete_predicate = f"target.id_ot IN ({ids_predicate_string})"
    # --- End of new logic ---

    # The unique key for matching rows remains the same.
    unique_key_predicate = "target.id_ot = source.id_ot AND target.Item = source.Item"

    (dt.merge(
            source=modified_df,
            predicate=unique_key_predicate,
            source_alias="source",
            target_alias="target"
        )
        .when_matched_update_all()  # Rule 1: If a row exists, update it.
        .when_not_matched_insert_all()  # Rule 2: If it's a new row, insert it.
        .when_not_matched_by_source_delete(  # Rule 3: If an old row is now gone...
            predicate=delete_predicate  # ...delete it, but ONLY if it belongs to an OT we are modifying.
        )
        .execute()
    )
    saved = "✅ **Successfully saved changes for all modified OTs to Delta Lake!**"
except Exception as e:
    saved = f"❌ **Error saving to Delta Lake:** {e}"

print(saved)

dt = DeltaTable(table_path)
df = dt.to_pandas()
print(f"VERSION Actual del Deltalake: {dt.version()}")


## <span style="color: cyan; font-size: 26px;">**GENERAR REPORTE EXCEL BASE** para todas las cuadrillas</span>


### Desde `df` Organiza, recarga avances y crea `base_he_{today}`

In [27]:
# Creación he entre fechas
# Genera un DataFrame entre las fechas para proceder a la exportacion
he = df.copy()

# ── DataFrame conditioning ──────────────────────────────────────────────────────────
he['Fecha'] = he['Fecha'].apply(lambda x: eerssa.utils.soloFecha_SinTimezone( x ))
he['Date']  = he['Fecha'].apply(lambda x: eerssa.utils.toDateObject( x ))
he['InicioEvento'] = he['InicioEvento'].apply( lambda x: eerssa.utils.elimina_timezone( x ) )
he['FinEvento'] = he['FinEvento'].apply( lambda x: eerssa.utils.elimina_timezone( x ) )
he['Duracion'] = he['Duracion']/(60*24)

he = he[ 
    (he['Date'] >= reporte_inicia   ) &   # Fecha Inicial
    (he['Date'] <= reporte_finaliza ) &   # Fecha Final
    (he['HorasExtra']== 'Si' )            # Es Hora Extra
    ]

he = he[[ 'Cuadrilla', 'Responsable', 'Dia', 'Date','Item', 'InicioEvento', 'FinEvento', 'Duracion', 'Evento','Cuenta','id_ot','Archivo' ]]


In [17]:
he

,Cuadrilla,Responsable,Dia,Date,Item,InicioEvento,FinEvento,Duracion,Evento,Cuenta,id_ot,Archivo
1,Yacuambi Z1 (Cuadrilla. Nro. 8),"NL, NR, WC",jueves,2026-04-23,8,2026-04-23 17:30:00,2026-04-23 18:10:00,0.027778,En Conchay se encuentra Med 197261 sin servici...,ACOMETIDAS,175403,OT [03] Cuadrilla Yacuambi 2026-04-23 (017) NL...
3,Yacuambi Z1 (Cuadrilla. Nro. 8),"NL, NR",jueves,2026-04-23,15,2026-04-23 20:53:00,2026-04-23 22:00:00,0.046528,En Guaguayme Bajo en la estructura 28712 se en...,REDES,175403,OT [03] Cuadrilla Yacuambi 2026-04-23 (017) NL...
6,Yacuambi Z1 (Cuadrilla. Nro. 8),"NL, NR",jueves,2026-04-23,14,2026-04-23 19:53:00,2026-04-23 20:53:00,0.041667,DAÑO REPORTADO POR Centro de Control. Mensaje ...,transporte,175403,OT [03] Cuadrilla Yacuambi 2026-04-23 (017) NL...
7,Yacuambi Z1 (Cuadrilla. Nro. 8),"NL, NR",jueves,2026-04-23,20,2026-04-23 22:00:00,2026-04-23 23:59:00,0.082639,Desde Guaguayme Alto se retorna a Yacuambi,transporte,175403,OT [03] Cuadrilla Yacuambi 2026-04-23 (017) NL...
9,Yacuambi Z1 (Cuadrilla. Nro. 8),"NL, NR, WC",jueves,2026-04-23,10,2026-04-23 18:10:00,2026-04-23 19:13:00,0.043750,Desde Conchay se retorna a Yacuambi,transporte,175403,OT [03] Cuadrilla Yacuambi 2026-04-23 (017) NL...
...,...,...,...,...,...,...,...,...,...,...,...,...
1903,Yacuambi Z1 (Cuadrilla. Nro. 8),LOZANO SIGCHO NAUN ENRIQUE,martes,2026-04-21,12,2026-04-21 00:00:01,2026-04-21 00:00:02,0.000000,"SE LABORA: NL, NR y WC de 08:00 a 13:00 y de 1...",se_labora,175200,OT [03] Cuadrilla Yacuambi 2026-04-21 (017) NL...
1916,Zamora Z1 (Cuadrilla. AP Nro. 4),MORALES RIVERA LUIS ALBERTO,martes,2026-04-21,17,2026-04-21 00:00:01,2026-04-21 00:00:02,0.000000,"SE LABORA, LM, RY, JCH y MC, desde las 08:00 a...",se_labora,175167,OT [02] Alumbrado Zamora 2026-04-21 (012) LM.pdf
1928,El Pangui Z1 (Cuadrilla. Nro. 4),MENDIETA MENDIETA HENRRY ALEXANDER,martes,2026-04-21,13,2026-04-21 00:00:01,2026-04-21 00:00:02,0.000000,"SE LABORA: HM, LV, AD de 8:00 a 13:15 y de 14:...",se_labora,175170,OT [08] Cuadrilla El Pangui 2026-04-21 (042) H...
1939,Guayzimi Z1 (Cuadrilla. Nro. 7),QUIROGA ORDONEZ CARLOS HERNAN,martes,2026-04-21,15,2026-04-21 00:00:01,2026-04-21 00:00:02,0.000000,"SE LABORA: CQ, AC, JL de: 08:00 a 13:00 y de 1...",se_labora,175177,OT [07] Cuadrilla Guayzimi 2026-04-21 (038) CQ...


In [28]:
# Agrupa y Etiqueta 
# Parse datetime columns
he['InicioEvento'] = pd.to_datetime(he['InicioEvento'])
he['FinEvento'] = pd.to_datetime(he['FinEvento'])

# Truncate to the minute
he['Inicio_min'] = he['InicioEvento'].dt.floor('min')
he['Fin_min'] = he['FinEvento'].dt.floor('min')

# Sort
he = he.sort_values(['id_ot', 'InicioEvento']).reset_index(drop=True)

# Detect breaks: new group starts when id_ot changes OR there's a time gap
new_group = (
    (he['id_ot'] != he['id_ot'].shift()) |
    (he['Inicio_min'] != he['Fin_min'].shift())
)

# Assign group number using cumulative sum of breaks
he['group'] = new_group.cumsum()

# Agrupacion de filas colapsando los 'Eventos'
result = he.groupby(['group']).agg(
    Cuadrilla    = ('Cuadrilla', 'first'),
    Responsable  = ('Responsable', 'first'),
    Dia          = ('Dia','first'),
    Date         = ('Date','first'),
    InicioEvento = ('InicioEvento', 'min'),    # Start of the group
    FinEvento    = ('FinEvento', 'max'),       # End of the group
    Duracion     = ('Duracion', 'sum'),        # Total duration
    Evento       = ('Evento', list),           # All events in the group
    Cuenta       = ('Cuenta', list),
    id_ot        = ('id_ot','first'),
    Items        = ('Item',list),
    Num_Filas    = ('Evento', 'count'),         # How many rows in the group
    Archivo      = ('Archivo','first')
).reset_index().drop(columns='group').query('Duracion != 0.0').sort_values(['Archivo', 'InicioEvento']).reset_index(drop=True)

# Limpiar el texto y limpiar las cuentas
result['Evento'] = result['Evento'].apply(lambda x: eerssa.utils.limpiar_lista_eventos(x))
result['Cuenta'] = result['Cuenta'].apply(lambda x: eerssa.utils.limpiar_cuentas(x))
result['Cuenta'] = result['Cuenta'].apply(eerssa.utils.cuenta_to_dict) # Crea diccionarios de las cuentas
result['Items'] = result['Items'].apply(lambda x: eerssa.utils.limpiar_items(x))

#result['Items'] = result['Items'].apply(lambda x: str(sorted(x)) if isinstance(x, list) else str(x) )
result[['InicioEvento', 'FinEvento']] = result[['InicioEvento', 'FinEvento']].apply(lambda x: x.dt.time) #TimeObjects

# Create the formula string for every row, starting at index 2
result['Duracion'] = [f'=F{i}-E{i}' for i in range(2, len(result) + 2)]



# AGREGAR ETIQUETAS DE TIPO DE HORA EXTRA
# ---------------------------------------------------------
# Se cargan desde la Plantilla de Horas Extra. 
# TODO: Para desacoplar este código necesito Hydra con las 
# variables globales: he_template_path 

# visualizar dias festivos
festivos = pd.read_excel(
    he_template_path,
    sheet_name='Revisar_primero',
    usecols='A:C',      # Only columns A and B
    header=0            # First row as column names
)

# visualizar cambios de horario en Cuadrilla Alumbrado
noche = pd.read_excel(
    he_template_path,
    sheet_name='Revisar_primero',
    usecols='D',      # Only columns A and B
    header=0            # First row as column names
).squeeze("columns") # Turns the 1-column DataFrame into a Series


# 4. PREPARACIÓN DE DICCIONARIOS Y SETS (BÚSQUEDAS RÁPIDAS)
# ---------------------------------------------------------
festivos['Fecha'] = pd.to_datetime(festivos['Fecha']).dt.date

dict_todos = festivos[festivos['Aplica'] == 'TODOS'].set_index('Fecha')['Etiqueta'].to_dict()
dict_especificos = festivos[festivos['Aplica'] != 'TODOS'].set_index(['Fecha', 'Aplica'])['Etiqueta'].to_dict()

# Limpiamos la serie 'noche', extraemos solo la fecha y la convertimos en un Set
set_noche = set(pd.to_datetime(noche.dropna()).dt.date)

# Constante para la cuadrilla especial
CUADRILLA_AP_4 = "Zamora Z1 (Cuadrilla. AP Nro. 4)"


# 5. DEFINICIÓN DE FUNCIONES DE LÓGICA DE NEGOCIO
# ---------------------------------------------------------

def obtener_etiqueta_festivo(fecha, cuadrilla):
    """Retorna la etiqueta del festivo. Prioriza específicos locales sobre los nacionales."""
    if (fecha, cuadrilla) in dict_especificos:
        return dict_especificos[(fecha, cuadrilla)]
    if fecha in dict_todos:
        return dict_todos[fecha]
    return None

def clasificar_tipo(row):
    """Evalúa las reglas de negocio para determinar el tipo de remuneración/jornada."""
    fecha     = row['Date']
    cuadrilla = row['Cuadrilla']
    dia       = row['Dia']
    inicio    = row['InicioEvento']

    # 1. CAMBIO_HORARIO — Caso especial exclusivo para Alumbrado Público
    if cuadrilla == CUADRILLA_AP_4 and fecha in set_noche:
        return 'CAMBIO_HORARIO'

    # 2. FESTIVO — Verificamos diccionarios generales
    if fecha in dict_todos:
        return 'FESTIVO'

    # 3. CANTONIZACION — Verificamos diccionarios específicos
    if (fecha, cuadrilla) in dict_especificos:
        return 'CANTONIZACION'

    # 4. DESCANSO — Fines de semana
    if dia in ('sábado', 'domingo'):
        return 'DESCANSO'

    # 5. MAD (Madrugada) — Si el trabajo inició antes de las 6:00 AM
    if inicio.hour < 6:
        return 'MAD'

    # 6. NORMAL — Valor por defecto
    return 'NORMAL'


result['Tipo'] = result.apply(clasificar_tipo, axis=1)



In [14]:
result

,Cuadrilla,Responsable,Dia,Date,InicioEvento,FinEvento,Duracion,Evento,Cuenta,id_ot,Items,Num_Filas,Archivo,Tipo
0,Jefatura Zonal Zamora,EP,viernes,2026-04-24,16:30:00,18:47:00,=F2-E2,Grupo de Línea Energizada realiza el cambio de...,"[{'cuenta': 'REDES', 'peso': 1.0}]",175603,"['8', '10']",2,OT [00] 00_Jefe Zonal 2026-04-24 (004) EP.pdf,NORMAL
1,Jefatura Zonal Zamora,EP,sábado,2026-04-25,07:00:00,18:00:00,=F3-E3,"En la Subestación Gualaquiza, se coordina con ...","[{'cuenta': 'REDES', 'peso': 1.0}]",175605,"['2', '5', '6']",3,OT [00] 00_Jefe Zonal 2026-04-25 (004) EP.pdf,DESCANSO
2,Jefatura Zonal Zamora,EP,domingo,2026-04-26,07:00:00,13:00:00,=F4-E4,Se coordina con Centro de Control y empresa SE...,"[{'cuenta': 'REDES', 'peso': 1.0}]",175610,['1'],1,OT [00] 00_Jefe Zonal 2026-04-26 (004) EP.pdf,DESCANSO
3,Jefatura Zonal Zamora,EP,domingo,2026-04-26,14:00:00,17:00:00,=F5-E5,Se continua trabajos sobre el alimentador Bomb...,"[{'cuenta': 'REDES', 'peso': 1.0}]",175610,['5'],1,OT [00] 00_Jefe Zonal 2026-04-26 (004) EP.pdf,DESCANSO
4,Jefatura Zonal Zamora,EP,domingo,2026-04-26,18:00:00,20:45:00,=F6-E6,Retorno desde Gualaquiza hasta Zamora,"[{'cuenta': '?', 'peso': 1.0}]",175610,['7'],1,OT [00] 00_Jefe Zonal 2026-04-26 (004) EP.pdf,DESCANSO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169,Gualaquiza (Agencia),"CB, GC",viernes,2026-04-03,08:53:00,13:15:00,=F171-E171,"Gualaquiza, recepcion reclamo Centro Control, ...","[{'cuenta': 'MEDIDORES', 'peso': 1.0}]",174080,"['1', '2', '3', '4', '5']",5,OT [24] Agencia Gualaquiza 2026-04-03 (061) CB...,FESTIVO
170,Gualaquiza (Agencia),"CB, GC",viernes,2026-04-03,17:01:00,19:05:00,=F172-E172,"Traslado Flor del Bosque, reclamo Centro Contr...","[{'cuenta': 'REDES', 'peso': 1.0}]",174080,"['9', '10', '11', '12', '13']",5,OT [24] Agencia Gualaquiza 2026-04-03 (061) CB...,FESTIVO
171,Gualaquiza (Agencia),"CB, GC",sábado,2026-04-04,16:16:00,18:24:00,=F173-E173,"Gualaquiza, recepcion reclamo Centro Control, ...","[{'cuenta': 'REDES', 'peso': 1.0}]",174083,"['1', '2', '3']",3,OT [24] Agencia Gualaquiza 2026-04-04 (061) CB...,DESCANSO
172,Gualaquiza (Agencia),"CB, GC",domingo,2026-04-05,08:48:00,10:49:00,=F174-E174,"Gualaquiza, recepcion reclamo Centro Control, ...","[{'cuenta': 'REDES', 'peso': 1.0}]",174084,"['1', '2', '3']",3,OT [24] Agencia Gualaquiza 2026-04-05 (061) CB...,DESCANSO


In [29]:
# Ajuste de Horarios Inicio Fin

result['InicioEvento'] = result.apply(eerssa.utils.ajustar_horario_inicio, axis=1)
result['FinEvento'] = result.apply(eerssa.utils.ajustar_horario_fin, axis=1)

#TODO: Dia Festivo delante del evento, antes de la id_ot

result['Evento'] = result.apply(
    lambda row: f"OT # {row['id_ot']}. {row['Evento']}", 
    axis=1
)   


In [224]:
result

,Cuadrilla,Responsable,Dia,Date,InicioEvento,FinEvento,Duracion,Evento,Cuenta,id_ot,Items,Num_Filas,Archivo,Tipo
0,Zamora Z1 (Cuadrilla. Nro. 6),"FR, JCR",miércoles,2026-04-01,22:31:00,23:33:00,=F2-E2,OT # 173886. DAÑO REPORTADO POR Centro de Cont...,"[{'cuenta': 'REDES', 'peso': 1.0}]",173886,[10],1,OT [01] Cuadrilla Zamora 2026-04-01 (006) FR.pdf,NORMAL
1,Zamora Z1 (Cuadrilla. Nro. 6),"FR, JCR",viernes,2026-04-03,09:13:00,12:53:00,=F3-E3,"OT # 174090. En la Agencia Zamora, se coordina...","[{'cuenta': 'REDES', 'peso': 1.0}]",174090,"[2, 3, 5, 6, 8]",5,OT [01] Cuadrilla Zamora 2026-04-03 (006) FR.pdf,FESTIVO
2,Zamora Z1 (Cuadrilla. Nro. 6),"FR, JCR",viernes,2026-04-03,15:38:00,18:05:00,=F4-E4,OT # 174090. Desde la Agencia Zamora nos trasl...,"[{'cuenta': 'ACOMETIDAS', 'peso': 1.0}]",174090,"[13, 14, 15]",3,OT [01] Cuadrilla Zamora 2026-04-03 (006) FR.pdf,FESTIVO
3,Zamora Z1 (Cuadrilla. Nro. 6),"RS, JCR",sábado,2026-04-04,20:40:00,21:41:00,=F5-E5,OT # 174606. Nos trasladamos desde la agencia ...,"[{'cuenta': 'REDES', 'peso': 1.0}]",174606,"[2, 3, 4]",3,OT [01] Cuadrilla Zamora 2026-04-04 (007) RS.pdf,DESCANSO
4,Zamora Z1 (Cuadrilla. Nro. 6),"FR, RS, LL, VZH, JCR",miércoles,2026-04-08,17:00:00,18:32:00,=F6-E6,OT # 174212. DAÑO REPORTADO POR Centro de Cont...,"[{'cuenta': 'REDES', 'peso': 1.0}]",174212,"[11, 12]",2,OT [01] Cuadrilla Zamora 2026-04-08 (006) FR.pdf,NORMAL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57,Gualaquiza (Agencia),"CB, GC",jueves,2026-04-02,19:17:00,20:46:00,=F59-E59,"OT # 173960. Traslado Guayusal. Guayusal, reiv...","[{'cuenta': 'REDES', 'peso': 1.0}]",173960,"[16, 17, 18]",3,OT [24] Agencia Gualaquiza 2026-04-02 (061) CB...,NORMAL
58,Gualaquiza (Agencia),"CB, GC",viernes,2026-04-03,08:53:00,13:15:00,=F60-E60,"OT # 174080. Gualaquiza, recepcion reclamo Cen...","[{'cuenta': 'MEDIDORES', 'peso': 1.0}]",174080,"[1, 2, 3, 4, 5]",5,OT [24] Agencia Gualaquiza 2026-04-03 (061) CB...,FESTIVO
59,Gualaquiza (Agencia),"CB, GC",viernes,2026-04-03,17:01:00,19:05:00,=F61-E61,"OT # 174080. Traslado Flor del Bosque, reclamo...","[{'cuenta': 'REDES', 'peso': 1.0}]",174080,"[9, 10, 11, 12, 13]",5,OT [24] Agencia Gualaquiza 2026-04-03 (061) CB...,FESTIVO
60,Gualaquiza (Agencia),"CB, GC",sábado,2026-04-04,16:16:00,18:24:00,=F62-E62,"OT # 174083. Gualaquiza, recepcion reclamo Cen...","[{'cuenta': 'REDES', 'peso': 1.0}]",174083,"[1, 2, 3]",3,OT [24] Agencia Gualaquiza 2026-04-04 (061) CB...,DESCANSO


In [30]:
# DIVIDIR los festivos en LUNCH

# Condición combinada
mask = (
    result['Tipo'].isin(["DESCANSO", "FESTIVO", "CANTONIZACION"]) &
    (result['FinEvento'] > time(15, 0, 0)) & (result['InicioEvento'] < time(12, 0, 0)) &
    ((pd.to_datetime(result['FinEvento'].astype(str)) - pd.to_datetime(result['InicioEvento'].astype(str))).dt.total_seconds() > 4 * 3600)
)

# Filas que NO cumplen la condición (se mantienen igual)
no_split = result[~mask].copy()

# Filas que SÍ cumplen la condición
to_split = result[mask].copy()

# Primera parte: original con FinEvento = 13:00
part1 = to_split.copy()
part1['FinEvento'] = time(13, 0, 0)
part1['Items'] = "1, 2" # para diferenciarlos en DB_HE_2026

# Segunda parte: duplicada con InicioEvento = 14:00
part2 = to_split.copy()
part2['InicioEvento'] = time(14, 0, 0)
part2['Items'] = "3, 4" # para diferenciarlos en DB_HE_2026

# Concatenar todo y resetear índice
result = pd.concat([no_split, part1, part2], ignore_index=True).sort_values(['Archivo', 'InicioEvento']).reset_index(drop=True)

# Vuelve a recalcular las formulas de Duración
result['Duracion'] = [f'=F{i}-E{i}' for i in range(2, len(result) + 2)]


/tmp/ipykernel_464524/174788964.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ((pd.to_datetime(result['FinEvento'].astype(str)) - pd.to_datetime(result['InicioEvento'].astype(str))).dt.total_seconds() > 4 * 3600)
/tmp/ipykernel_464524/174788964.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ((pd.to_datetime(result['FinEvento'].astype(str)) - pd.to_datetime(result['InicioEvento'].astype(str))).dt.total_seconds() > 4 * 3600)


<span style="color: oragne; font-size: 22px;">**RECARGAR EVENTOS** En caso de existir recarga los eventos y Cuentas ya editados </span>

Depende de dos variables: `id_ot` y `Num_Filas` cuando se dividen las filas se colocan nuevos identificadores de filas. 


In [31]:
# Recargar Pickl y repopular 'Evento' y 'Cuenta'
result = eerssa.utils.download_he_db( result, db_he_path )


>>> [download] Copied values into 176 matching rows.


In [18]:
db = pd.read_pickle( db_he_path )
db

,Cuadrilla,Responsable,Dia,Fecha,InicioEvento,FinEvento,Duracion,Evento,Cuenta,id_ot,Items,Num_Filas,Archivo,Tipo
0,Jefatura Zonal Zamora,EP,viernes,2026-04-24,17:00:00,18:47:00,=F2-E2,"OT # 175603. Chuchumbletza, apoyo a Grupo de L...","[{'cuenta': 'REDES', 'peso': 1.0}]",175603,"['8', '10']",2.0,OT [00] 00_Jefe Zonal 2026-04-24 (004) EP.pdf,NORMAL
1,Jefatura Zonal Zamora,EP,sábado,2026-04-25,07:00:00,13:00:00,=F3-E3,"OT # 175605. Tunduli, se coordina con personal...","[{'cuenta': 'REDES', 'peso': 1.0}]",175605,"1, 2",3.0,OT [00] 00_Jefe Zonal 2026-04-25 (004) EP.pdf,DESCANSO
2,Jefatura Zonal Zamora,EP,sábado,2026-04-25,14:00:00,18:00:00,=F4-E4,"OT # 175605. Tunduli, paso de cuerda manila en...","[{'cuenta': 'REDES', 'peso': 1.0}]",175605,"3, 4",3.0,OT [00] 00_Jefe Zonal 2026-04-25 (004) EP.pdf,DESCANSO
3,Jefatura Zonal Zamora,EP,domingo,2026-04-26,07:00:00,13:00:00,=F5-E5,"OT # 175610. Tunduli, se coordina con personal...","[{'cuenta': 'REDES', 'peso': 1.0}]",175610,['1'],1.0,OT [00] 00_Jefe Zonal 2026-04-26 (004) EP.pdf,DESCANSO
4,Jefatura Zonal Zamora,EP,domingo,2026-04-26,14:00:00,17:00:00,=F6-E6,"OT # 175610. Tunduli, paso de cuerda manila en...","[{'cuenta': 'REDES', 'peso': 1.0}]",175610,['5'],1.0,OT [00] 00_Jefe Zonal 2026-04-26 (004) EP.pdf,DESCANSO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143,Gualaquiza (Agencia),"CB, GC",viernes,2026-04-03,08:53:00,13:15:00,=F145-E145,OT # 174080. DIA FESTIVO “Viernes Santo” Naich...,"[{'cuenta': 'MEDIDORES', 'peso': 1.0}]",174080,"['1', '2', '3', '4', '5']",5.0,OT [24] Agencia Gualaquiza 2026-04-03 (061) CB...,FESTIVO
144,Gualaquiza (Agencia),"CB, GC",viernes,2026-04-03,17:01:00,19:05:00,=F146-E146,"OT # 174080. Flor del Bosque, Est 273993, revi...","[{'cuenta': 'REDES', 'peso': 1.0}]",174080,"['9', '10', '11', '12', '13']",5.0,OT [24] Agencia Gualaquiza 2026-04-03 (061) CB...,FESTIVO
145,Gualaquiza (Agencia),"CB, GC",sábado,2026-04-04,16:16:00,18:24:00,=F147-E147,"OT # 174083. Guayusal, revision LMV estructura...","[{'cuenta': 'REDES', 'peso': 1.0}]",174083,"['1', '2', '3']",3.0,OT [24] Agencia Gualaquiza 2026-04-04 (061) CB...,DESCANSO
146,Gualaquiza (Agencia),"CB, GC",domingo,2026-04-05,08:48:00,10:49:00,=F148-E148,"OT # 174084. Chumpias, revision LMV estructura...","[{'cuenta': 'REDES', 'peso': 1.0}]",174084,"['1', '2', '3']",3.0,OT [24] Agencia Gualaquiza 2026-04-05 (061) CB...,DESCANSO


In [19]:
result

,Cuadrilla,Responsable,Dia,Date,InicioEvento,FinEvento,Duracion,Evento,Cuenta,id_ot,Items,Num_Filas,Archivo,Tipo
0,Jefatura Zonal Zamora,EP,viernes,2026-04-24,17:00:00,18:47:00,=F2-E2,"OT # 175603. Chuchumbletza, apoyo a Grupo de L...","[{'cuenta': 'REDES', 'peso': 1.0}]",175603,"['8', '10']",2,OT [00] 00_Jefe Zonal 2026-04-24 (004) EP.pdf,NORMAL
1,Jefatura Zonal Zamora,EP,sábado,2026-04-25,07:00:00,13:00:00,=F3-E3,"OT # 175605. Tunduli, se coordina con personal...","[{'cuenta': 'REDES', 'peso': 1.0}]",175605,"1, 2",3,OT [00] 00_Jefe Zonal 2026-04-25 (004) EP.pdf,DESCANSO
2,Jefatura Zonal Zamora,EP,sábado,2026-04-25,14:00:00,18:00:00,=F4-E4,"OT # 175605. Tunduli, paso de cuerda manila en...","[{'cuenta': 'REDES', 'peso': 1.0}]",175605,"3, 4",3,OT [00] 00_Jefe Zonal 2026-04-25 (004) EP.pdf,DESCANSO
3,Jefatura Zonal Zamora,EP,domingo,2026-04-26,07:00:00,13:00:00,=F5-E5,"OT # 175610. Tunduli, se coordina con personal...","[{'cuenta': 'REDES', 'peso': 1.0}]",175610,['1'],1,OT [00] 00_Jefe Zonal 2026-04-26 (004) EP.pdf,DESCANSO
4,Jefatura Zonal Zamora,EP,domingo,2026-04-26,14:00:00,17:00:00,=F6-E6,"OT # 175610. Tunduli, paso de cuerda manila en...","[{'cuenta': 'REDES', 'peso': 1.0}]",175610,['5'],1,OT [00] 00_Jefe Zonal 2026-04-26 (004) EP.pdf,DESCANSO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,Gualaquiza (Agencia),"CB, GC",viernes,2026-04-03,08:53:00,13:15:00,=F175-E175,OT # 174080. DIA FESTIVO “Viernes Santo” Naich...,"[{'cuenta': 'MEDIDORES', 'peso': 1.0}]",174080,"['1', '2', '3', '4', '5']",5,OT [24] Agencia Gualaquiza 2026-04-03 (061) CB...,FESTIVO
174,Gualaquiza (Agencia),"CB, GC",viernes,2026-04-03,17:01:00,19:05:00,=F176-E176,"OT # 174080. Flor del Bosque, Est 273993, revi...","[{'cuenta': 'REDES', 'peso': 1.0}]",174080,"['9', '10', '11', '12', '13']",5,OT [24] Agencia Gualaquiza 2026-04-03 (061) CB...,FESTIVO
175,Gualaquiza (Agencia),"CB, GC",sábado,2026-04-04,16:16:00,18:24:00,=F177-E177,"OT # 174083. Guayusal, revision LMV estructura...","[{'cuenta': 'REDES', 'peso': 1.0}]",174083,"['1', '2', '3']",3,OT [24] Agencia Gualaquiza 2026-04-04 (061) CB...,DESCANSO
176,Gualaquiza (Agencia),"CB, GC",domingo,2026-04-05,08:48:00,10:49:00,=F178-E178,"OT # 174084. Chumpias, revision LMV estructura...","[{'cuenta': 'REDES', 'peso': 1.0}]",174084,"['1', '2', '3']",3,OT [24] Agencia Gualaquiza 2026-04-05 (061) CB...,DESCANSO


### EXPORTAR a  Excel Base de Horas Extra

In [20]:
from openpyxl.styles import Font, PatternFill, Border, Side
from openpyxl.styles.differential import DifferentialStyle
from openpyxl.formatting.rule import Rule

# ============================================================
# STYLE DEFINITIONS
# ============================================================

# 1. Red background, dark red text (Light Red Fill with Dark Red Text)
style_FinEvento = DifferentialStyle(
    font=Font(color="9C0006"),
    fill=PatternFill(bgColor="FFC7CE")
)

# 2. Yellow background, dark yellow text
style_sabado = DifferentialStyle(
    font=Font(color="000000"),
    fill=PatternFill(bgColor="FFEB9C")
)

# 2. Yellow background, dark yellow text
style_domingo = DifferentialStyle(
    font=Font(color="000000"),
    fill=PatternFill(bgColor="EEE31D")
)


# 3. Green background, dark green text
style_green = DifferentialStyle(
    font=Font(color="006100"),
    fill=PatternFill(bgColor="C6EFCE")
)

# 4. Dark red background, white text
style_red_error = DifferentialStyle(
    font=Font(color="FFFFFF"),
    fill=PatternFill(bgColor="C00000")
)

# 5. Red text, no fill
style_red_text = DifferentialStyle(
    font=Font(color="000000"),
    fill=PatternFill(bgColor="FF9D9D")
)

# 6. Bold black text, no fill
style_bold = DifferentialStyle(
    font=Font(bold=True)
)

# 7. Blue background, white text
style_blue = DifferentialStyle(
    font=Font(color="FFFFFF"),
    fill=PatternFill(bgColor="4472C4")
)


# 7. Blue background, white text
style_mad = DifferentialStyle(
    font=Font(color="000000"),
    fill=PatternFill(bgColor="97B4E8")
)


# 8. White text, purple background
style_festivo = DifferentialStyle(
    font=Font(color="FFFFFF"),
    fill=PatternFill(bgColor="7030A0")
)


style_purple = DifferentialStyle(
    font=Font(color="FFFFFF"),
    fill=PatternFill(bgColor="FF8D8DB3")
)



# 9. White text, gray background
style_gray = DifferentialStyle(
    font=Font(color="FFFFFF"),
    fill=PatternFill(bgColor="808080")
)

# 10. Thick black bottom border
style_bottom_border = DifferentialStyle(
    border=Border(
        bottom=Side(style="thick", color="000000")
    )
)

# ============================================================
# REGLAS DE FORMATO CONDICIONAL CORREGIDAS
# ============================================================

rules_informe_he = [
    # 1. Regla fines de semana y Festivos (Púrpura)
    # Usamos WEEKDAY($B6, 2) > 5 para detectar Sábados (6) y Domingos (7)
    # Usamos SEARCH para que no importe si dice "Festivo" o "FESTIVO"
    {
        "formula": '=OR(WEEKDAY($B6, 2) > 5, ISNUMBER(SEARCH("FESTIVO", $J6)))',
        "style": style_purple,
        "range": "A6:J36",
    },

    # 2. Regla solo para festivos (Negrita)
    {
        "formula": '=ISNUMBER(SEARCH("FESTIVO", $J6))',
        "style": style_bold,
        "range": "A6:I36"
    }
]


rules = [
    # Rule 1 - Red bg: =CONTAR.SI(Revisar_primero!$B$2:$B$14, $D1)>0
    {
        "formula": '=COUNTIF(Revisar_primero!$B$2:$B$14,$D1)>0',
        "style": style_festivo,
        "range": "A1:N5000",
    },
    # Rule 2 - Yellow bg: =Y( O($A1="Zamora Z1 (Cuadrilla. Nro. 6)", $A1="Zamora Z1 (Cuadrilla. AP Nro. 4)", $A1="Zamora (Agencia)", $A1="Jefatura Zonal Zamora"),  CONTAR.SI(Revisar_primero!$B$15:$B$18, $D1)>0 )
    {
        "formula": '=AND(OR($A1="Zamora Z1 (Cuadrilla. Nro. 6)",$A1="Zamora Z1 (Cuadrilla. AP Nro. 4)",$A1="Zamora (Agencia)",$A1="Jefatura Zonal Zamora"),COUNTIF(Revisar_primero!$B$15:$B$18,$D1)>0)',
        "style": style_festivo,
        "range": "A1:N5000",
    },
    # Rule 3 - Yellow bg: =Y( $A1="Yacuambi Z1 (Cuadrilla. Nro. 8)",  CONTAR.SI(Revisar_primero!$B$19:$B$19, $D1)>0 )
    {
        "formula": '=AND($A1="Yacuambi Z1 (Cuadrilla. Nro. 8)",COUNTIF(Revisar_primero!$B$19:$B$19,$D1)>0)',
        "style": style_festivo,
        "range": "A1:N5000",
    },
    # Rule 4 - Yellow bg: =Y( O($A1="Yantzaza Z1 (Cuadrilla. Nro. 5)", $A1="Lineas Energizadas (Cuadrilla Nro.6)", $A1="Yantzaza (Agencia)"),  CONTAR.SI(Revisar_primero!$B$20:$B$22, $D1)>0 )
    {
        "formula": '=AND(OR($A1="Yantzaza Z1 (Cuadrilla. Nro. 5)",$A1="Lineas Energizadas (Cuadrilla Nro.6)",$A1="Yantzaza (Agencia)"),COUNTIF(Revisar_primero!$B$20:$B$22,$D1)>0)',
        "style": style_festivo,
        "range": "A1:N5000",
    },
    # Rule 5 - Yellow bg: =Y( $A1="Paquisha Z1 (Cuadrilla. Nro. 10)",  CONTAR.SI(Revisar_primero!$B$23:$B$23, $D1)>0 )
    {
        "formula": '=AND($A1="Paquisha Z1 (Cuadrilla. Nro. 10)",COUNTIF(Revisar_primero!$B$23:$B$23,$D1)>0)',
        "style": style_festivo,
        "range": "A1:N5000",
    },
    # Rule 6 - Yellow bg: =Y( $A1="Guayzimi Z1 (Cuadrilla. Nro. 7)",  CONTAR.SI(Revisar_primero!$B$24:$B$24, $D1)>0 )
    {
        "formula": '=AND($A1="Guayzimi Z1 (Cuadrilla. Nro. 7)",COUNTIF(Revisar_primero!$B$24:$B$24,$D1)>0)',
        "style": style_festivo,
        "range": "A1:N5000",
    },
    # Rule 7 - Yellow bg: =Y( O($A1="El Pangui Z1 (Cuadrilla. Nro. 4)", $A1="El Pangui (Agencia)"),  CONTAR.SI(Revisar_primero!$B$25:$B$26, $D1)>0 )
    {
        "formula": '=AND(OR($A1="El Pangui Z1 (Cuadrilla. Nro. 4)",$A1="El Pangui (Agencia)"),COUNTIF(Revisar_primero!$B$25:$B$26,$D1)>0)',
        "style": style_festivo,
        "range": "A1:N5000",
    },
    # Rule 8 - Yellow bg: =Y( O($A1="Gualaquiza Z1 (Cuadrilla. Nro. 3)", $A1="Gualaquiza (Agencia)"),  CONTAR.SI(Revisar_primero!$B$27:$B$28, $D1)>0 )
    {
        "formula": '=AND(OR($A1="Gualaquiza Z1 (Cuadrilla. Nro. 3)",$A1="Gualaquiza (Agencia)"),COUNTIF(Revisar_primero!$B$27:$B$28,$D1)>0)',
        "style": style_festivo,
        "range": "A1:N5000",
    },
    # Rule 10 - Bold: =$C1="sábado"
    {
        "formula": '=$C1="sábado"',
        "style": style_sabado,
        "range": "A1:N5000",
    },
    # Rule 11 - Dark red bg: =$C1="domingo"
    {
        "formula": '=$C1="domingo"',
        "style": style_domingo,
        "range": "A1:N5000",
    },
    # Rule 14 - Blue bg: =Y($A1="Zamora Z1 (Cuadrilla. AP Nro. 4)", CONTAR.SI(Revisar_primero!$D$2:$D$114, $D1)>0)
    {
        "formula": '=AND($A1="Zamora Z1 (Cuadrilla. AP Nro. 4)",COUNTIF(Revisar_primero!$D$2:$D$114,$D1)>0)',
        "style": style_blue,
        "range": "A1:N5000",
    },
    
    # Rule 9 - Green bg: =$A1<>$A2
    {
        "formula": "=$A1<>$A2",
        "style": style_bottom_border,
        "range": "A1:N5000",
    },
    
    # Rule 12 - Red text: =$E1<>"" * (HORA($E1)<6 * (HORA($F1) + MINUTO($F1)/100) > 8)
    {
        "formula": '=AND($E1<>"",(HOUR($E1)<6),((HOUR($F1)+MINUTE($F1)/100)>6))',
        "style": style_red_error,
        "range": "A1:N5000",
    },
    # Rule 13 - Gray bg: =$F1<$E1
    {
        "formula": "=$F1<$E1",
        "style": style_red_error,
        "range": "A1:N5000",
    },
    # Rule 15 - Purple bg: =$E1<>"" Y (HORA($E1)<6)
    {
        "formula": '=AND($E1<>"",HOUR($E1)<6)',
        "style": style_mad,
        "range": "A1:N5000",
    },
    # Rule 16 - Bottom border: =Y(HORA($E1)<8, (HORA($F1)+MINUTO($F1)/100)>8)
    {
        "formula": "=AND(HOUR($E1)<8,(HOUR($F1)+MINUTE($F1)/100)>8)",
        "style": style_red_text,
        "range": "F1:F5000",
    },
    # Rule 17 - Bottom border: =Y(HORA($E1)>17, HORA($F1)>17)
    {
        "formula": "=AND(HOUR($E1)<17,HOUR($F1)>17)",
        "style": style_red_text,
        "range": "E1:E5000",
    },
]

# FUNCTION TO APPLY RULES TO WORKSHEET
# ============================================================

def apply_conditional_formatting(ws):
    """Apply all conditional formatting rules to a worksheet."""
    for r in rules:
        rule = Rule(
            type="expression",
            formula=[r["formula"]],
            dxf=r["style"],
            stopIfTrue=False,
        )
        ws.conditional_formatting.add(r["range"], rule)


def apply_he_informe(ws):
    """Apply all conditional formatting rules to a worksheet."""
    for r in rules_informe_he:
        rule = Rule(
            type="expression",
            formula=[r["formula"]],
            dxf=r["style"],
            stopIfTrue=False,
        )
        ws.conditional_formatting.add(r["range"], rule)



In [32]:
# Genera el archivo de Excel "base_he_2026XXX" desde el Dataset `result`

# Copia el Template
shutil.copyfile(he_template_path, output_he)
print(f"Se ha copiado Excel para HE en {output_he}")

# Load the copied workbook
wb = load_workbook(output_he)

# Access the second sheet (0-based index; change if needed)
sheet = wb.worksheets[1]  # Or wb['result'] if named

# Optional: Clear existing data from row 2 down (preserves headers and formats)
#for row in sheet.iter_rows(min_row=2, max_row=sheet.max_row, min_col=1, max_col=sheet.max_column):
#    for cell in row:
#        cell.value = None


# ===== Manejo de CUENTAS 

result['Cuenta'] = result['Cuenta'].apply( lambda x: eerssa.utils.dict_to_cuenta(x) )


# Insert DataFrame starting from row 2 (skip headers in DF)
for r_idx, row in enumerate(dataframe_to_rows(result, index=False, header=False), 2):
    for c_idx, value in enumerate(row, 1):
        sheet.cell(row=r_idx, column=c_idx, value=value)

apply_conditional_formatting( sheet )

# Save the modified workbook
wb.save(output_he)
print(f"✅ Se ha generado el archivo de EXCEL en -> {output_he}")


Se ha copiado Excel para HE en reporte/base_HE_20260504.xlsx
✅ Se ha generado el archivo de EXCEL en -> reporte/base_HE_20260504.xlsx


## <span style="color: red; font-size: 32px;">**RECARGAR EXCEL** para cargar de Base Horas Extra Excel -> Pandas -> Pickl</span>

### Leer Excel `base_he_{today}` 

In [ ]:
# Para pruebas desde archivo ya generado
# output_he   = "reporte/base_HE_20260422.xlsx"

In [33]:
# 1.  >> AQUI << LECTURA DE DATOS DESDE EXCEL
# ---------------------------------------------------------
# data_only=False es CRÍTICO para obtener el texto de la fórmula (ej. '=F2-E2') 
# en lugar de su resultado calculado en Excel.

#wb = load_workbook('reporte/base_HE_20260417.xlsx', data_only=False)

wb = load_workbook(output_he, data_only=False)
ws = wb["result"]
data = ws.values
cols = next(data)  
base = pd.DataFrame(data, columns=cols)
base = base.dropna(subset=['Cuadrilla'])

#base['Items'] = base['Items'].apply(lambda x: str(sorted(x)) if isinstance(x, list) else str(x))

# Convierte los valores de texto plano a diccionario
base['Cuenta'] = base['Cuenta'].apply(eerssa.utils.cuenta_to_dict)
base['id_ot']  = base['id_ot'].apply( lambda x: int(x) )

# Estandarizar Fechas y Horas a objetos nativos
base['Fecha'] = pd.to_datetime(base['Fecha']).dt.date
base['InicioEvento'] = pd.to_datetime(base['InicioEvento'], format='%H:%M:%S').dt.time
base['FinEvento'] = pd.to_datetime(base['FinEvento'], format='%H:%M:%S').dt.time

/home/vlad/GIT/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


In [186]:
base

,Cuadrilla,Responsable,Dia,Fecha,InicioEvento,FinEvento,Duracion,Evento,Cuenta,id_ot,Items,Num_Filas,Archivo,Tipo
0,Jefatura Zonal Zamora,EP,viernes,2026-04-24,17:00:00,18:47:00,=F2-E2,"OT # 175603. Chuchumbletza, apoyo a Grupo de L...","[{'cuenta': 'REDES', 'peso': 1.0}]",175603,"['8', '10']",2.0,OT [00] 00_Jefe Zonal 2026-04-24 (004) EP.pdf,NORMAL
1,Jefatura Zonal Zamora,EP,sábado,2026-04-25,07:00:00,13:00:00,=F3-E3,"OT # 175605. Tunduli, se coordina con personal...","[{'cuenta': 'REDES', 'peso': 1.0}]",175605,"1, 2",3.0,OT [00] 00_Jefe Zonal 2026-04-25 (004) EP.pdf,DESCANSO
2,Jefatura Zonal Zamora,EP,sábado,2026-04-25,14:00:00,18:00:00,=F4-E4,"OT # 175605. Tunduli, paso de cuerda manila en...","[{'cuenta': 'REDES', 'peso': 1.0}]",175605,"3, 4",3.0,OT [00] 00_Jefe Zonal 2026-04-25 (004) EP.pdf,DESCANSO
3,Jefatura Zonal Zamora,EP,domingo,2026-04-26,07:00:00,13:00:00,=F5-E5,"OT # 175610. Tunduli, se coordina con personal...","[{'cuenta': 'REDES', 'peso': 1.0}]",175610,['1'],1.0,OT [00] 00_Jefe Zonal 2026-04-26 (004) EP.pdf,DESCANSO
4,Jefatura Zonal Zamora,EP,domingo,2026-04-26,14:00:00,17:00:00,=F6-E6,"OT # 175610. Tunduli, paso de cuerda manila en...","[{'cuenta': 'REDES', 'peso': 1.0}]",175610,['5'],1.0,OT [00] 00_Jefe Zonal 2026-04-26 (004) EP.pdf,DESCANSO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143,Gualaquiza (Agencia),"CB, GC",viernes,2026-04-03,08:53:00,13:15:00,=F145-E145,OT # 174080. DIA FESTIVO “Viernes Santo” Naich...,"[{'cuenta': 'MEDIDORES', 'peso': 1.0}]",174080,"['1', '2', '3', '4', '5']",5.0,OT [24] Agencia Gualaquiza 2026-04-03 (061) CB...,FESTIVO
144,Gualaquiza (Agencia),"CB, GC",viernes,2026-04-03,17:01:00,19:05:00,=F146-E146,"OT # 174080. Flor del Bosque, Est 273993, revi...","[{'cuenta': 'REDES', 'peso': 1.0}]",174080,"['9', '10', '11', '12', '13']",5.0,OT [24] Agencia Gualaquiza 2026-04-03 (061) CB...,FESTIVO
145,Gualaquiza (Agencia),"CB, GC",sábado,2026-04-04,16:16:00,18:24:00,=F147-E147,"OT # 174083. Guayusal, revision LMV estructura...","[{'cuenta': 'REDES', 'peso': 1.0}]",174083,"['1', '2', '3']",3.0,OT [24] Agencia Gualaquiza 2026-04-04 (061) CB...,DESCANSO
146,Gualaquiza (Agencia),"CB, GC",domingo,2026-04-05,08:48:00,10:49:00,=F148-E148,"OT # 174084. Chumpias, revision LMV estructura...","[{'cuenta': 'REDES', 'peso': 1.0}]",174084,"['1', '2', '3']",3.0,OT [24] Agencia Gualaquiza 2026-04-05 (061) CB...,DESCANSO


In [83]:
# Visualizar la base de datos
db = pd.read_pickle( db_he_path)
db

,Cuadrilla,Responsable,Dia,Fecha,InicioEvento,FinEvento,Duracion,Evento,Cuenta,id_ot,Items,Num_Filas,Archivo,Tipo
0,Jefatura Zonal Zamora,EP,viernes,2026-04-24,17:00:00,18:47:00,=F2-E2,"OT # 175603. Chuchumbletza, apoyo a Grupo de L...","[{'cuenta': 'REDES', 'peso': 1.0}]",175603,"['8', '10']",2.0,OT [00] 00_Jefe Zonal 2026-04-24 (004) EP.pdf,NORMAL
1,Jefatura Zonal Zamora,EP,sábado,2026-04-25,07:00:00,13:00:00,=F3-E3,"OT # 175605. Tunduli, se coordina con personal...","[{'cuenta': 'REDES', 'peso': 1.0}]",175605,"1, 2",3.0,OT [00] 00_Jefe Zonal 2026-04-25 (004) EP.pdf,DESCANSO
2,Jefatura Zonal Zamora,EP,sábado,2026-04-25,14:00:00,18:00:00,=F4-E4,"OT # 175605. Tunduli, paso de cuerda manila en...","[{'cuenta': 'REDES', 'peso': 1.0}]",175605,"3, 4",3.0,OT [00] 00_Jefe Zonal 2026-04-25 (004) EP.pdf,DESCANSO
3,Jefatura Zonal Zamora,EP,domingo,2026-04-26,07:00:00,13:00:00,=F5-E5,"OT # 175610. Tunduli, se coordina con personal...","[{'cuenta': 'REDES', 'peso': 1.0}]",175610,['1'],1.0,OT [00] 00_Jefe Zonal 2026-04-26 (004) EP.pdf,DESCANSO
4,Jefatura Zonal Zamora,EP,domingo,2026-04-26,14:00:00,17:00:00,=F6-E6,"OT # 175610. Tunduli, paso de cuerda manila en...","[{'cuenta': 'REDES', 'peso': 1.0}]",175610,['5'],1.0,OT [00] 00_Jefe Zonal 2026-04-26 (004) EP.pdf,DESCANSO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,Gualaquiza (Agencia),"CB, GC",viernes,2026-04-03,08:53:00,13:15:00,=F137-E137,OT # 174080. DIA FESTIVO “Viernes Santo” Naich...,"[{'cuenta': 'MEDIDORES', 'peso': 1.0}]",174080,"['1', '2', '3', '4', '5']",5.0,OT [24] Agencia Gualaquiza 2026-04-03 (061) CB...,FESTIVO
136,Gualaquiza (Agencia),"CB, GC",viernes,2026-04-03,17:01:00,19:05:00,=F138-E138,"OT # 174080. Flor del Bosque, Est 273993, revi...","[{'cuenta': 'REDES', 'peso': 1.0}]",174080,"['9', '10', '11', '12', '13']",5.0,OT [24] Agencia Gualaquiza 2026-04-03 (061) CB...,FESTIVO
137,Gualaquiza (Agencia),"CB, GC",sábado,2026-04-04,16:16:00,18:24:00,=F139-E139,"OT # 174083. Guayusal, revision LMV estructura...","[{'cuenta': 'REDES', 'peso': 1.0}]",174083,"['1', '2', '3']",3.0,OT [24] Agencia Gualaquiza 2026-04-04 (061) CB...,DESCANSO
138,Gualaquiza (Agencia),"CB, GC",domingo,2026-04-05,08:48:00,10:49:00,=F140-E140,"OT # 174084. Chumpias, revision LMV estructura...","[{'cuenta': 'REDES', 'peso': 1.0}]",174084,"['1', '2', '3']",3.0,OT [24] Agencia Gualaquiza 2026-04-05 (061) CB...,DESCANSO


### Actualizar PICKLE_DB

In [34]:
# UPLOAD DB
# Exportamos al archivo existente en 'db_he_path'
# base.to_pickle(db_he_path)  # solo la primera vez
eerssa.utils.upload_he_db( base, db_he_path )

>>> [upload] Updated: 176 rows | Appended: 1 rows | >>> DB total: 187 rows.


### Dividir las HE para cada persona.

In [35]:
# Dividir para cada persona

# Convertir strings separados por comas de vuelta a listas nativas de Python
for col in ['Responsable']: 
    base[col] = base[col].apply(lambda x: str(x).split(", ") if pd.notnull(x) else [])

# 1. Extract all unique labels (initials) from base lists
unique_responsables = base['Responsable'].explode().dropna().unique()

# 3. Initialize a dictionary to map names to DataFrames
horasExtra_todos = {}

# Multiplier map for each Tipo
TIPO_MULTIPLIER = {
    "NORMAL":          1.5,
    "MAD":             2.0,
    "FESTIVO":         2.0,
    "DESCANSO":        2.0,
    "CANTONIZACION":   2.0,
    "CAMBIO_HORARIO":  0.25,
}

# Funcion que Pivotea los horarios
def pivot_time_events(filtered_df):
    df = filtered_df.copy()
    df['_occurrence'] = df.groupby('Fecha').cumcount()
    df = df[df['_occurrence'] < 3].copy()

    for i in range(1, 4):
        df[f'Extra_{i}'] = np.nan
        df[f'Fin_{i}'] = np.nan

    result_rows = []
    for fecha, group in df.groupby('Fecha', sort=False):
        group = group.sort_values('InicioEvento').reset_index(drop=True)
        base_row = group.iloc[0].copy()
        for i, (_, row) in enumerate(group.iterrows()):
            if i < 3:
                base_row[f'Extra_{i+1}'] = row['InicioEvento']
                base_row[f'Fin_{i+1}'] = row['FinEvento']

        # Aggregate Tipo values in order into a list
        base_row['Lista_tipos'] = group['Tipo'].tolist()

        result_rows.append(base_row)

    result = pd.DataFrame(result_rows)

    cols = result.columns.tolist()
    inicio_pos = cols.index('InicioEvento')
    new_cols = ['Extra_1', 'Fin_1', 'Extra_2', 'Fin_2', 'Extra_3', 'Fin_3']
    cols_clean = [c for c in cols if c not in new_cols + ['InicioEvento', 'FinEvento', '_occurrence']]
    final_cols = cols_clean[:inicio_pos] + new_cols + cols_clean[inicio_pos:]

    return result[final_cols].reset_index(drop=True)


def build_sobretiempos_formula(lista_tipos, row_i, fila_offset=6):
    """
    Builds an Excel SUM formula for overtime based on Lista_tipos.
    Each tipo maps to a multiplier. Pairs are (C,D), (E,F), (G,H).
    """
    col_pairs = [('C', 'D'), ('E', 'F'), ('G', 'H')]
    xl_row = row_i + fila_offset

    terms = []

    for idx, tipo in enumerate(lista_tipos):
        if idx >= 3:
            break

        tipo = str(tipo).strip()
        multiplier = TIPO_MULTIPLIER.get(tipo, 1.5)

        start_col, end_col = col_pairs[idx]
        terms.append(f"({end_col}{xl_row}-{start_col}{xl_row})*{multiplier}*24")

    if not terms:
        return ''

    return f'=SUM({",".join(terms)})'

# Funcion que rellena las fechas
def complete_date_range(df, reporte_inicia, reporte_finaliza):
    """
    Completa el DataFrame con todas las fechas entre reporte_inicia y reporte_finaliza.
    Las fechas sin datos se rellenan con NaN excepto la columna 'Fecha'.
    """
    all_dates = pd.DataFrame({
        'Fecha': pd.date_range(reporte_inicia, reporte_finaliza)
    })

    df['Fecha'] = pd.to_datetime(df['Fecha'])
    result = all_dates.merge(df, on='Fecha', how='left')

    return result

# 4. Loop through each unique person
for person in unique_responsables:

    # 5. Filter base DataFrame to keep only rows where the person is in the list
    filtered_df = base[
        base['Responsable'].apply(lambda x: person in x if isinstance(x, list) else False)
    ].copy()

    # 6. Reset index so rows are numbered from 0
    filtered_df = filtered_df.sort_values(by=['Fecha', 'InicioEvento'])
    filtered_df = filtered_df.reset_index(drop=True)

    # 7. PIVOTEAR los horarios
    filtered_df = pivot_time_events(filtered_df)

    # 8. Rellenar las fechas vacias
    filtered_df = complete_date_range(filtered_df, reporte_inicia, reporte_finaliza)

    # 9. Recalculate formulas based on FINAL row positions
    fila_inicial_xl = 6

    filtered_df['Duracion'] = [
        f'=SUM(D{i + fila_inicial_xl}-C{i + fila_inicial_xl}, F{i + fila_inicial_xl}-E{i + fila_inicial_xl}, H{i + fila_inicial_xl}-G{i + fila_inicial_xl})'
        for i in range(len(filtered_df))
    ]

    filtered_df['Sobretiempos'] = [
        build_sobretiempos_formula(lista_tipos, i, fila_inicial_xl)
        if isinstance(lista_tipos, list) else ''
        for i, lista_tipos in enumerate(filtered_df['Lista_tipos'], start=0)
    ]

    # 10. Store the filtered DataFrame
    horasExtra_todos[person] = filtered_df[
        [
            "Dia",
            "Fecha",
            "Extra_1",
            "Fin_1",
            "Extra_2",
            "Fin_2",
            "Extra_3",
            "Fin_3",
            "Duracion",
            "Evento",
            #"Lista_tipos",
            "Sobretiempos",
        ]
    ]

In [36]:
horasExtra_todos['JJ']

,Dia,Fecha,Extra_1,Fin_1,Extra_2,Fin_2,Extra_3,Fin_3,Duracion,Evento,Sobretiempos
0,NaN,2026-04-01,NaN,NaN,NaN,NaN,NaN,NaN,"=SUM(D6-C6, F6-E6, H6-G6)",NaN,
1,NaN,2026-04-02,NaN,NaN,NaN,NaN,NaN,NaN,"=SUM(D7-C7, F7-E7, H7-G7)",NaN,
2,NaN,2026-04-03,NaN,NaN,NaN,NaN,NaN,NaN,"=SUM(D8-C8, F8-E8, H8-G8)",NaN,
3,NaN,2026-04-04,NaN,NaN,NaN,NaN,NaN,NaN,"=SUM(D9-C9, F9-E9, H9-G9)",NaN,
4,NaN,2026-04-05,NaN,NaN,NaN,NaN,NaN,NaN,"=SUM(D10-C10, F10-E10, H10-G10)",NaN,
5,NaN,2026-04-06,NaN,NaN,NaN,NaN,NaN,NaN,"=SUM(D11-C11, F11-E11, H11-G11)",NaN,
6,NaN,2026-04-07,NaN,NaN,NaN,NaN,NaN,NaN,"=SUM(D12-C12, F12-E12, H12-G12)",NaN,
7,NaN,2026-04-08,NaN,NaN,NaN,NaN,NaN,NaN,"=SUM(D13-C13, F13-E13, H13-G13)",NaN,
8,jueves,2026-04-09,17:00:00,19:30:00,NaN,NaN,NaN,NaN,"=SUM(D14-C14, F14-E14, H14-G14)","OT # 174423. Santa Elena, Est 34357 Trafo 1444...",=SUM((D14-C14)*1.5*24)
9,viernes,2026-04-10,06:00:00,08:00:00,17:00:00,18:00:00,NaN,NaN,"=SUM(D15-C15, F15-E15, H15-G15)","OT # 174471. Guaysimi, se recoge a los compañe...","=SUM((D15-C15)*1.5*24,(F15-E15)*1.5*24)"


<span style="color: cyan; font-size: 22px;">**informe excel** para cada uno</span>

POR HACER: 

- ~~Obtener datos de Nombre y Cuadrilla desde GDrive import~~
- ~~Calcular el total de Horas con Sobretiempos y sobreescribir la formula~~
- ~~Cargar las reglas condicionales desde OpenPyXL.~~
  


In [37]:
from eerssa import organizar as gdrive    # download sheet from Google Drive
df_datos_cuadrilla = gdrive.get_gsheet_df() 
df_datos_cuadrilla

INFO:eerssa.organizar:Authentication successful!
INFO:eerssa.organizar:|->> Successfully opened Google Sheet: 'DB_calificar_ot'
INFO:eerssa.organizar:|->> Selected worksheet by name: 'Iniciales'
INFO:eerssa.organizar:
|->> Data successfully imported into DataFrame


,ORDEN_RESPONSABLE,NOMBRE,INICIALES,CUADRILLA_OT,CUADRILLA_CORTO,ORDEN_CUADRILLA
0,002,MARQUEZ APOLO JHONNY FABIAN,JM,Geico,Geico,00
1,003,QUEZADA ORDOEZ ANDREW ISRAEL,AQ,IN1,IN1,00
2,004,PALACIOS MERINO ERNESTO VLADIMIR,EP,Jefatura Zonal Zamora,Jefe Zonal Zamora,00
4,006,RIOS RIOS FRANCISCO FERNANDO,FR,Zamora Z1 (Cuadrilla. Nro. 6),Cuadrilla Zamora,01
5,007,SILVA ARMIJOS ROMEL EDUARDO,RS,Zamora Z1 (Cuadrilla. Nro. 6),Cuadrilla Zamora,01
6,008,LEON CUEVA LUIS ALBERTO,LL,Zamora Z1 (Cuadrilla. Nro. 6),Cuadrilla Zamora,01
7,009,ZHUNAULA GUAMAN VICTOR ANIBAL,VZH,Zamora Z1 (Cuadrilla. Nro. 6),Cuadrilla Zamora,01
8,010,RIVERA GUAMAN SEGUNDO PATRICIO,SR,Zamora Z1 (Cuadrilla. Nro. 6),Cuadrilla Zamora,01
9,011,CASTILLO RODRIGUEZ JIMMY LEODAN,JCR,Zamora Z1 (Cuadrilla. Nro. 6),Cuadrilla Zamora,01
10,012,MORALES RIVERA LUIS ALBERTO,LM,Zamora Z1 (Cuadrilla. AP Nro. 4),Alumbrado Zamora,02


In [38]:
# Generar archivo de todos con Horas Extra


import shutil
import pandas as pd
from openpyxl import load_workbook


# ── Paths ──────────────────────────────────────────────────────────────────────
TEMPLATE_PATH = "models/he_2026_plantilla.xlsx"
OUTPUT_PATH = "reporte/testing.xlsx"
TEMPLATE_SHEET = "HORAS_EXTRA"
START_ROW = 6
START_COL = 2  # Column B

# ── Copy template to output ────────────────────────────────────────────────────
shutil.copy2(TEMPLATE_PATH, OUTPUT_PATH)

# ── Load workbook copy ─────────────────────────────────────────────────────────
wb = load_workbook(OUTPUT_PATH)
template_ws = wb[TEMPLATE_SHEET]

# ── Create one sheet per dataframe in the dictionary ───────────────────────────
for key, df in horasExtra_todos.items():
    # Copy template sheet
    new_ws = wb.copy_worksheet(template_ws)
    new_ws.title = str(key)
    
    # ── Consultar datos de la cuadrilla ────────────────────────────────────────
    # Filtramos el DataFrame donde la columna INICIALES coincide con 'key'
    match = df_datos_cuadrilla[df_datos_cuadrilla['INICIALES'] == key]
    
    # Si encontramos una coincidencia, extraemos los datos y los insertamos
    if not match.empty:
        nombre_completo = match['NOMBRE'].iloc[0]
        nombre_cuadrilla = match['CUADRILLA_CORTO'].iloc[0]
        
        new_ws['H2'] = nombre_completo
        new_ws['H3'] = nombre_cuadrilla
    else:
        # Fallback en caso de que las iniciales no existan en el CSV
        new_ws['H2'] = "No registrado"
        new_ws['H3'] = "No registrado"

    # Omit first dataframe column
    df_to_export = df.iloc[:, 1:]

    # Write data starting at row 6, column B
    for row_idx, row_data in enumerate(df_to_export.itertuples(index=False), start=START_ROW):
        for col_idx, value in enumerate(row_data, start=START_COL):
            new_ws.cell(row=row_idx, column=col_idx, value=value)

    # Configurar el pie de página a la derecha
    # &P es el código para la página actual
    # &N es el código para el total de páginas
    new_ws.oddFooter.right_footer =  '&"Arial,Regular"&10 Página &P de &N'
    new_ws.evenFooter.right_footer =  '&"Arial,Regular"&10 Página &P de &N'

    # 2. Asegurar que los márgenes le den espacio a LibreOffice para dibujarlo
    # (Los valores están en pulgadas)
    new_ws.page_margins.bottom = 0.55  # Margen inferior general de la página
    new_ws.page_margins.footer = 0.42   # Espacio específico reservado para el pie de página

    apply_he_informe(new_ws)

# Optional: remove template sheet from output if you don't want it included
# del wb[TEMPLATE_SHEET]

wb.save(OUTPUT_PATH)
print(f"Workbook saved to: {OUTPUT_PATH}")

Workbook saved to: reporte/testing.xlsx


# Legacy Horas Extra
puedes ser util para Gustavo Torres

Generar archivo de todos con Horas Extra = SIN FORMATO =   GTORRES

```python
# Use Pandas ExcelWriter with the openpyxl engine
with pd.ExcelWriter('reporte/he_final_abril.xlsx', engine='openpyxl') as writer:
    
    for person, df in horasExtra_todos.items():
        
        df_sorted = df.copy()

        # 2. Write to Excel
        df_sorted.to_excel(writer, sheet_name=person, index=False)
        
        # 3. Access the underlying openpyxl worksheet
        worksheet = writer.sheets[person]

        # 4. Get column indexes (1-based)
        fecha_col_idx    = df_sorted.columns.get_loc('Fecha') + 1
        evento_col_idx   = df_sorted.columns.get_loc('Evento') + 1
        duration_col_idx   = df_sorted.columns.get_loc('Duracion') + 1
        
        j1_Ini_col_idx = df_sorted.columns.get_loc('Extra_1') + 1
        j1_fin_col_idx = df_sorted.columns.get_loc('Fin_1') + 1
        j2_Ini_col_idx = df_sorted.columns.get_loc('Extra_2') + 1
        j2_fin_col_idx = df_sorted.columns.get_loc('Fin_2') + 1
        j3_Ini_col_idx = df_sorted.columns.get_loc('Extra_3') + 1
        j3_fin_col_idx = df_sorted.columns.get_loc('Fin_3') + 1
        
        evento_col_letter = get_column_letter(evento_col_idx)

        # 5. Auto-fit all columns based on content, except Evento
        for col in worksheet.columns:
            col_letter = get_column_letter(col[0].column)
            if col_letter == evento_col_letter:
                worksheet.column_dimensions[col_letter].width = 103  # ~14cm
            else:
                max_length = max(
                    (len(str(cell.value)) if cell.value is not None else 0)
                    for cell in col
                )
                worksheet.column_dimensions[col_letter].width = min(max_length + 2, 60)

        # 6. Format Dates and Times
        # Note: 'hh:mm' is standard for clock times, '[h]:mm' allows hours to go over 24 for durations
        clock_time_format = 'hh:mm' 
        duration_format   = '[h]:mm'
        date_format       = 'dd/mm/yyyy'

        for row in range(2, len(df_sorted) + 2):  # Skip header row
            # Format Start and End Times
            worksheet.cell(row=row, column=j1_Ini_col_idx).number_format = clock_time_format
            worksheet.cell(row=row, column=j2_fin_col_idx).number_format = clock_time_format
            worksheet.cell(row=row, column=j2_Ini_col_idx).number_format = clock_time_format
            worksheet.cell(row=row, column=j2_fin_col_idx).number_format = clock_time_format
            worksheet.cell(row=row, column=j3_Ini_col_idx).number_format = clock_time_format
            worksheet.cell(row=row, column=j3_fin_col_idx).number_format = clock_time_format
            
            # Format Duration
            worksheet.cell(row=row, column=duration_col_idx).number_format = duration_format
            
            # Format Date
            worksheet.cell(row=row, column=fecha_col_idx).number_format = date_format
```

## exportar informe final
- Los tiempos personales se editan y ajustan los minutos en el informe final.

#### GENERAR Horas Extra individual

<span style="color: red; font-size: 22px;">**NOMBRE DEL ARCHIVO** para guardar de Base Horas Extra Excel a Pandas</span>

In [85]:
guardar_he_todos = os.path.join('reporte', 'Reporte_he_todos.xlsx')


In [ ]:
# Generar archivo de todos con Horas Extra

# Use Pandas ExcelWriter with the openpyxl engine
with pd.ExcelWriter(guardar_he_todos, engine='openpyxl') as writer:
    
    for person, df in horasExtra_todos.items():
        
        # 1. Sort the DataFrame by Date and Time
        # reset_index(drop=True) is good practice here to clear the old row numbers
        df_sorted = df.sort_values(by=['Fecha','InicioEvento']).reset_index(drop=True)
        
        # FIX: Re-build the Excel formulas AFTER sorting so they match the new physical rows
        # Assuming InicioEvento is E and FinEvento is F. 
        # (If their positions changed, adjust the letters here)
        df_sorted['Duracion'] = [f'=F{i}-E{i}' for i in range(2, len(df_sorted) + 2)]
        
        # 2. Write to Excel
        df_sorted.to_excel(writer, sheet_name=person, index=False)
        
        # 3. Access the underlying openpyxl worksheet
        worksheet = writer.sheets[person]

        # 4. Get column indexes (1-based)
        evento_col_idx   = df_sorted.columns.get_loc('Evento') + 1
        duracion_col_idx = df_sorted.columns.get_loc('Duracion') + 1
        j2_fin_col_idx   = df_sorted.columns.get_loc('InicioEvento') + 1
        j3_Ini_col_idx      = df_sorted.columns.get_loc('FinEvento') + 1
        fecha_col_idx    = df_sorted.columns.get_loc('Fecha') + 1
        
        evento_col_letter = get_column_letter(evento_col_idx)

        # 5. Auto-fit all columns based on content, except Evento (fixed 14cm ≈ 53 units)
        for col in worksheet.columns:
            col_letter = get_column_letter(col[0].column)
            if col_letter == evento_col_letter:
                worksheet.column_dimensions[col_letter].width = 53  # ~14cm
            else:
                max_length = max(
                    (len(str(cell.value)) if cell.value is not None else 0)
                    for cell in col
                )
                worksheet.column_dimensions[col_letter].width = min(max_length + 2, 60)

        # 6. Format Dates and Times
        # Note: 'hh:mm' is standard for clock times, '[h]:mm' allows hours to go over 24 for durations
        clock_time_format = 'hh:mm' 
        duration_format   = '[h]:mm'
        date_format       = 'dd/mm/yyyy'

        for row in range(2, len(df_sorted) + 2):  # Skip header row
            # Format Start and End Times
            worksheet.cell(row=row, column=j2_fin_col_idx).number_format = clock_time_format
            worksheet.cell(row=row, column=j3_Ini_col_idx).number_format = clock_time_format
            
            # Format Duration
            worksheet.cell(row=row, column=duracion_col_idx).number_format = duration_format
            
            # Format Date
            worksheet.cell(row=row, column=fecha_col_idx).number_format = date_format
        
        # 7. Fecha merging logic
        fecha_col_idx = df_sorted.columns.get_loc('Fecha') + 1
        start_row = 2
        max_row = len(df_sorted) + 1
        current_date = worksheet.cell(row=start_row, column=fecha_col_idx).value

        for row in range(3, max_row + 2):
            cell_value = worksheet.cell(row=row, column=fecha_col_idx).value if row <= max_row else None

            if cell_value != current_date:
                if row - start_row > 1:
                    worksheet.merge_cells(
                        start_row=start_row,
                        start_column=fecha_col_idx,
                        end_row=row - 1,
                        end_column=fecha_col_idx
                    )
                    worksheet.cell(row=start_row, column=fecha_col_idx).alignment = Alignment(
                        horizontal='center', vertical='center'
                    )
                current_date = cell_value
                start_row = row

### Desde Excel individuales

In [87]:
# 1. Leer todas las hojas del archivo Excel modificado
# Al usar sheet_name=None, Pandas devuelve un diccionario temporal
# Formato: {'Nombre_Hoja_1': DataFrame1, 'Nombre_Hoja_2': DataFrame2, ...}
todas_las_hojas = pd.read_excel(guardar_he_todos, sheet_name=None)

# 2. Inicializar el diccionario final
horasExtra_validado = {}

# 3. Iterar sobre cada hoja para limpiar los datos y guardarlos
for person, df in todas_las_hojas.items():
    
    # Solo procesar si el DataFrame no está vacío
    if not df.empty:
        
        # --- ARREGLO DE CELDAS COMBINADAS (MERGED CELLS) ---
        # Pandas lee la primera celda combinada y deja el resto como NaN.
        # ffill() (forward fill) arrastra la fecha hacia abajo para llenar esos NaNs.
        if 'Fecha' in df.columns:
            df['Fecha'] = df['Fecha'].ffill()
            
            # Asegurarse de que vuelva a ser un objeto Date puro nativo
            df['Fecha'] = pd.to_datetime(df['Fecha']).dt.date
        
        # --- RESTAURACIÓN DE FORMATOS DE HORA ---
        # Al leer de Excel, nos aseguramos de que las columnas de hora vuelvan
        # a ser objetos datetime.time de Python para futuros cálculos.
        if 'InicioEvento' in df.columns:
            df['InicioEvento'] = pd.to_datetime(
                df['InicioEvento'], format='%H:%M:%S', errors='coerce'
            ).dt.time
            
        if 'FinEvento' in df.columns:
            df['FinEvento'] = pd.to_datetime(
                df['FinEvento'], format='%H:%M:%S', errors='coerce'
            ).dt.time

        # 4. Guardar el DataFrame limpio en el nuevo diccionario
        # La clave (key) será el nombre de la hoja (person/cuadrilla)
        horasExtra_validado[person] = df

# Comprobación rápida para ver qué hojas se cargaron
print(f"Se cargaron exitosamente {len(horasExtra_validado)} cuadrillas en el diccionario.")
print("Claves disponibles:", list(horasExtra_validado.keys()))

Se cargaron exitosamente 40 cuadrillas en el diccionario.
Claves disponibles: ['FR', 'RS', 'LL', 'JCR', 'VZH', 'GJ', 'SO', 'LM', 'RY', 'JCH', 'MC', 'NL', 'WC', 'NR', 'SB', 'AA', 'AO', 'JLC', 'GT', 'RCL', 'PCH', 'LP', 'GLE', 'JA', 'DG', 'MS', 'JJ', 'CQ', 'AC', 'JL', 'HM', 'LV', 'AD', 'FA', 'AS', 'FG', 'CLP', 'MA', 'CB', 'GC']


In [91]:
horasExtra_validado['LV']

,Cuadrilla,Responsable,Dia,Fecha,InicioEvento,FinEvento,Duracion,Evento,Cuenta,id_ot,Items,Num_Filas,Archivo,Tipo
0,El Pangui Z1 (Cuadrilla. Nro. 4),"['LV', 'FA', 'AS']",domingo,2026-03-01,12:29:00,15:05:00,0 days 02:36:00,"*La Recta El Pangui*, estructura 272177 se enc...","[{'cuenta': 'REDES', 'peso': 1.0}]",171751,[],4,OT [08] Cuadrilla El Pangui 2026-03-01 (043) L...,DESCANSO
1,El Pangui Z1 (Cuadrilla. Nro. 4),"['LV', 'FA', 'AS']",domingo,2026-03-01,20:20:00,22:02:00,0 days 01:42:00,"*Chuchumbletza*, estructura 32025 transformado...","[{'cuenta': 'ACOMETIDAS', 'peso': 1.0}]",171751,[],3,OT [08] Cuadrilla El Pangui 2026-03-01 (043) L...,DESCANSO
2,El Pangui Z1 (Cuadrilla. Nro. 4),"['HM', 'LV', 'AS']",jueves,2026-03-05,17:00:00,18:03:00,0 days 01:03:00,"*Chantzas*, estructura 178710, Medidor 240186,...","[{'cuenta': 'ACOMETIDAS', 'peso': 1.0}]",172115,[],2,OT [08] Cuadrilla El Pangui 2026-03-05 (042) H...,NORMAL
3,El Pangui Z1 (Cuadrilla. Nro. 4),"['HM', 'LV', 'AD', 'AS']",martes,2026-03-10,17:00:00,19:16:00,0 days 02:16:00,*Traslado desde Soldado Rivera hasta El Pangui...,"[{'cuenta': 'ACOMETIDAS', 'peso': 1.0}]",172409,[],3,OT [08] Cuadrilla El Pangui 2026-03-10 (042) H...,NORMAL
4,El Pangui Z1 (Cuadrilla. Nro. 4),"['LV', 'FA']",sábado,2026-03-14,01:04:00,02:05:00,0 days 01:01:00,"*El Pangui*, estructura 27990 transformador de...","[{'cuenta': 'REDES', 'peso': 1.0}]",172693,[],3,OT [08] Cuadrilla El Pangui 2026-03-14 (043) L...,DESCANSO
5,El Pangui Z1 (Cuadrilla. Nro. 4),"['LV', 'FA', 'MS']",sábado,2026-03-14,07:53:00,17:41:00,0 days 09:48:00,"*Las Orquideas*, caminando desde donde llega e...","[{'cuenta': 'REDES', 'peso': 1.0}]",172693,[],6,OT [08] Cuadrilla El Pangui 2026-03-14 (043) L...,DESCANSO
6,El Pangui Z1 (Cuadrilla. Nro. 4),"['LV', 'FA']",domingo,2026-03-15,08:55:00,13:00:00,0 days 04:05:00,"*Traslado a Gualaquiza*, se colabora en arregl...","[{'cuenta': 'REDES', 'peso': 1.0}]",172909,[],7,OT [08] Cuadrilla El Pangui 2026-03-15 (043) L...,DESCANSO
7,El Pangui Z1 (Cuadrilla. Nro. 4),"['LV', 'FA']",domingo,2026-03-15,14:00:00,18:30:00,0 days 04:30:00,*Traslado desde Gualaquiza al sector El Mirado...,"[{'cuenta': 'REDES', 'peso': 1.0}]",172909,[],7,OT [08] Cuadrilla El Pangui 2026-03-15 (043) L...,DESCANSO
8,El Pangui Z1 (Cuadrilla. Nro. 4),"['HM', 'AS', 'LV']",martes,2026-03-17,17:00:00,18:09:00,0 days 01:09:00,"*Quebrada del Miassi*, estructura 27106 se des...","[{'cuenta': 'REDES', 'peso': 1.0}]",172875,[],2,OT [08] Cuadrilla El Pangui 2026-03-17 (042) H...,NORMAL
9,El Pangui Z1 (Cuadrilla. Nro. 4),"['HM', 'LV', 'AD']",miércoles,2026-03-25,17:00:00,18:38:00,0 days 01:38:00,"*Chuchumbletza*, estructura. *32026 se repone ...","[{'cuenta': 'REDES', 'peso': 1.0}]",173419,"['15', '16', '18']",3,OT [08] Cuadrilla El Pangui 2026-03-25 (042) H...,NORMAL


In [92]:
horasExtra_final = eerssa.utils.build_horas_extra_final(horasExtra_validado)

In [93]:
horasExtra_final['MA']

,Fecha,InicioEvento,FinEvento,Normal,Descanso,Madrugada,Evento
0,2026-03-02,17:00:00,18:05:00,=(C2-B2),,,"*San Francisco*, se revisa red de MV y se encu..."
1,2026-03-03,17:00:00,19:34:00,=(C3-B3),,,"*Sector Zapotillo*, se desbroza vegetación, se..."
2,2026-03-05,17:00:00,18:03:00,=(C4-B4),,,"*Las Peñas*, en la estructura 64727 se encuent..."
3,2026-03-06,19:19:00,20:39:00,=(C5-B5),,,"*El Belén*, en la estructura 127785 se revisa ..."
4,2026-03-07,11:22:00,15:21:00,,=(C6-B6),,"*El Belén*, domicilio de Gózalo Orellana, medi..."
5,2026-03-08,17:26:00,19:36:00,,=(C7-B7),,"*La Pradera*, en la estructura 265102 se revis..."
6,2026-03-12,17:15:00,19:26:00,=(C8-B8),,,"*El Rosario*, se realiza montaje del transform..."
7,2026-03-14,06:27:00,13:00:00,,=(C9-B9),,"*Luis Casiragui y Elias Brito*, RBT se constat..."
8,2026-03-14,14:00:00,16:07:00,,=(C10-B10),,"*La Pradera*, desbroce de maleza de la estruct..."
9,2026-03-14,18:23:00,19:27:00,,=(C11-B11),,"*Sevilla*, se revisa RBV conectores flojos en ..."


-----------------

In [ ]:
# Generar archivo de todos con Horas Extra

# Use Pandas ExcelWriter with the openpyxl engine
with pd.ExcelWriter('reporte/he_final.xlsx', engine='openpyxl') as writer:
    
    for person, df in horasExtra_final.items():
        
        df_sorted = df

        # 2. Write to Excel
        df_sorted.to_excel(writer, sheet_name=person, index=False)
        
        # 3. Access the underlying openpyxl worksheet
        worksheet = writer.sheets[person]

        # 4. Get column indexes (1-based)
        evento_col_idx   = df_sorted.columns.get_loc('Evento') + 1
        j1_Ini_col_idx = df_sorted.columns.get_loc('Normal') + 1
        j1_fin_col_idx = df_sorted.columns.get_loc('Descanso') + 1
        j2_Ini_col_idx = df_sorted.columns.get_loc('Madrugada') + 1
        j2_fin_col_idx   = df_sorted.columns.get_loc('InicioEvento') + 1
        j3_Ini_col_idx      = df_sorted.columns.get_loc('FinEvento') + 1
        fecha_col_idx    = df_sorted.columns.get_loc('Fecha') + 1
        
        evento_col_letter = get_column_letter(evento_col_idx)

        # 5. Auto-fit all columns based on content, except Evento (fixed 14cm ≈ 53 units)
        for col in worksheet.columns:
            col_letter = get_column_letter(col[0].column)
            if col_letter == evento_col_letter:
                worksheet.column_dimensions[col_letter].width = 103  # ~14cm
            else:
                max_length = max(
                    (len(str(cell.value)) if cell.value is not None else 0)
                    for cell in col
                )
                worksheet.column_dimensions[col_letter].width = min(max_length + 2, 60)

        # 6. Format Dates and Times
        # Note: 'hh:mm' is standard for clock times, '[h]:mm' allows hours to go over 24 for durations
        clock_time_format = 'hh:mm' 
        duration_format   = '[h]:mm'
        date_format       = 'dd/mm/yyyy'

        for row in range(2, len(df_sorted) + 2):  # Skip header row
            # Format Start and End Times
            worksheet.cell(row=row, column=j2_fin_col_idx).number_format = clock_time_format
            worksheet.cell(row=row, column=j3_Ini_col_idx).number_format = clock_time_format
            
            # Format Duration
            worksheet.cell(row=row, column=j1_Ini_col_idx).number_format = duration_format
            worksheet.cell(row=row, column=j1_fin_col_idx).number_format = duration_format
            worksheet.cell(row=row, column=j2_Ini_col_idx).number_format = duration_format
            
            # Format Date
            worksheet.cell(row=row, column=fecha_col_idx).number_format = date_format
        
        # 7. Fecha merging logic
        
        fecha_col_idx = df_sorted.columns.get_loc('Fecha') + 1
        start_row = 2
        max_row = len(df_sorted) + 1
        current_date = worksheet.cell(row=start_row, column=fecha_col_idx).value

        for row in range(3, max_row + 2):
            cell_value = worksheet.cell(row=row, column=fecha_col_idx).value if row <= max_row else None

            if cell_value != current_date:
                if row - start_row > 1:
                    worksheet.merge_cells(
                        start_row=start_row,
                        start_column=fecha_col_idx,
                        end_row=row - 1,
                        end_column=fecha_col_idx
                    )
                    worksheet.cell(row=start_row, column=fecha_col_idx).alignment = Alignment(
                        horizontal='center', vertical='center'
                    )
                current_date = cell_value
                start_row = row
        

<span style="color: BLUE; font-size: 26px;">**FIN DE REPORTES EN EXCEL** </span>


-----------------------------

# Full Mogno+Delta

In [22]:
import sys
import time
from pathlib import Path
import pandas as pd
import logging
import pymongo
from pymongo.errors import ConnectionFailure
from deltalake import DeltaTable, write_deltalake
from pprint import pprint
from datetime import datetime

from eerssa.secret import Keys

logging.basicConfig(level=logging.INFO)

DELTA_TABLE_PATH_ON_HOST = "/home/vlad/delta_V30"

table_path = DELTA_TABLE_PATH_ON_HOST


Success!!!


### Conexon con MongoDB

In [23]:
# --- MongoDB Connection ---
# It's better to establish the connection once and keep it open for the app's lifetime.
# We will also exit if the connection fails, as the consumer can't do its job without it.
uri = Keys.MONGO_KEY.value
client = None  # Initialize client to None
db_eerssa = None
CurrentCollection = None
ReloadCollection = None


try:
    # Add a timeout to avoid blocking indefinitely
    client = pymongo.MongoClient(uri, serverSelectionTimeoutMS=5000)
    # The ping command is cheap and does not require auth.
    client.admin.command('ping')
    db_eerssa = client.eerssa                   # Base de datos EERSSA
    CurrentCollection = db_eerssa.ot_v22        # Coleccion actual
    ReloadCollection  = db_eerssa.ot_reemplazo  # Aqui se cargan OTs repetidas
    logging.info(":::: Conexion exitosa con MongoDB ::::")
    
except ConnectionFailure as e:
    logging.error(f"\n\n ><><> Error de conexion a MongoDB: {e}")
    sys.exit(1) # Exit the script if we can't connect to MongoDB, as it's a critical dependency.


INFO:root::::: Conexion exitosa con MongoDB ::::


### Conexion con DeltaLake

In [24]:
# DELTA LAKE Connection

# Verify the existence of the DELTA LAKE table
if not DeltaTable.is_deltatable(table_path):
    print(
        f"No se ha encontrado la base de datos PARQUET-DELTALAKE en la direccion:\n NO_DELTA_LAKE : {table_path}" )
else:
    dt = DeltaTable(table_path)
    df = dt.to_pandas()
    print(f"Conectado a la tabla Delta Lake en: {table_path}")



Conectado a la tabla Delta Lake en: /home/vlad/delta_V30


In [25]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 411127 entries, 0 to 411126
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   Item           411127 non-null  int64 
 1   Cuenta         411127 non-null  object
 2   Evento         411127 non-null  object
 3   Actividad      411127 non-null  object
 4   Alimentador    411127 non-null  object
 5   Primario       411127 non-null  object
 6   Desconexion    411127 non-null  object
 7   SIG            411127 non-null  object
 8   Tipo           411127 non-null  object
 9   Materiales     411127 non-null  object
 10  Cuadrilla      411127 non-null  object
 11  Dia            411127 non-null  object
 12  Fecha          411127 non-null  object
 13  InicioEvento   411127 non-null  object
 14  FinEvento      411127 non-null  object
 15  Duracion       411127 non-null  int64 
 16  Responsable    411127 non-null  object
 17  Colaboradores  411127 non-null  int64 
 18  Hora

In [26]:
df

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
0,5,MEDIDORES,Chicaña se revisa medidor 19-207825 se encuen...,NO PROG,Los Encuentros,No,No,No,PREVENTIVO,·,...,2024-11-03 10:15:00,2024-11-03 10:55:00,40,OCHOA JARAMILLO ANGEL CLAUDIO,1,Si,4-100,YANTZAZA - MUTINZA - CHICAÑA - LA CETZA - MU...,140743,OT [04] Cuadrilla Yantzaza 2024-11-03 (032) AO...
1,1,informativa,DIA FESTIVO EN APEGO A LA LEY DE FERIADOS POR ...,INFO,·,No,No,No,·,·,...,2022-02-11 00:00:01,2022-02-11 00:00:02,0,MACAS CURIPOMA RAMIRO HOMERO,1,Si,2-110,Zamora,81013,OT [21] Agencia Zamora 2022-02-11 (027) RM.pdf
2,1,informativa,DIA FESTIVO EN APEGO A LA LEY DE FERIADOS POR ...,INFO,·,No,No,No,·,·,...,2022-02-11 00:00:01,2022-02-11 00:00:02,0,MACAS CURIPOMA RAMIRO HOMERO,1,Si,2-110,Zamora,81013,10_Test Orden de trabajo Zamora 11-02-2022 (RM...
3,1,informativa,DIA FESTIVO EN APEGO A LA LEY DE FERIADOS POR ...,INFO,·,No,No,No,·,·,...,2022-02-11 00:00:01,2022-02-11 00:00:02,0,MACAS CURIPOMA RAMIRO HOMERO,1,Si,2-110,Zamora,81013,OT [21] Agencia Zamora 2022-02-11 (027) RM.pdf
4,1,informativa,DIA FESTIVO EN APEGO A LA LEY DE FERIADOS POR ...,INFO,·,No,No,No,·,·,...,2022-02-11 00:00:01,2022-02-11 00:00:02,0,MACAS CURIPOMA RAMIRO HOMERO,1,Si,2-110,Zamora,81013,10_Test Orden de trabajo Zamora 11-02-2022 (RM...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
411122,1,informativa,DAÑO REPORTADO POR EL CENTRO DE CONTROL,INFO,·,No,No,No,·,·,...,2026-02-09 00:00:01,2026-02-09 00:00:02,0,SILVA ARMIJOS ROMEL EDUARDO,1,Si,2-110,Guadalupe,170615,OT [01] Cuadrilla Zamora 2026-02-09 (007) RS.pdf
411123,2,transporte,Nos trasladamos desde la agencia EERSSA Zamora...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2026-02-09 19:42:00,2026-02-09 20:20:00,38,SILVA ARMIJOS ROMEL EDUARDO,1,Si,2-110,Guadalupe,170615,OT [01] Cuadrilla Zamora 2026-02-09 (007) RS.pdf
411124,3,REDES,"Guadalupe, La Libertad en la estructura. 25893...",NO PROG,Yacuambi,No,No,No,CORRECTIVO,·,...,2026-02-09 20:20:00,2026-02-09 21:45:00,85,SILVA ARMIJOS ROMEL EDUARDO,1,Si,2-110,Guadalupe,170615,OT [01] Cuadrilla Zamora 2026-02-09 (007) RS.pdf
411125,4,transporte,Nos trasladamos desde La Libertad a la agencia...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2026-02-09 21:45:00,2026-02-09 22:19:00,34,SILVA ARMIJOS ROMEL EDUARDO,1,Si,2-110,Guadalupe,170615,OT [01] Cuadrilla Zamora 2026-02-09 (007) RS.pdf


### Analisis de Cambios en una OT

In [36]:
# recargar Deltalake

dt = DeltaTable(table_path)
df = dt.to_pandas()

In [37]:
#  ANALIZAR UNA OT filtrando por su indice

filtered_df = df.query( f"id_ot == 175857" ).sort_values(by='Item')
filtered_df


,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
0,1,informativa,"DIA FESTIVO. ""DÍA DEL TRABAJADOR""",INFO,·,No,No,No,·,·,...,2026-05-01 00:00:01,2026-05-01 00:00:02,0.0,LEON CUEVA LUIS ALBERTO,1,Si,2-110,Zamora,175857,OT [01] Cuadrilla Zamora 2026-05-01 (008) LL.pdf
386001,2,?,En Zamora se atiende daño reportado por Centro...,NO PROG,Zamora I,No,No,No,PREDICTIVO,·,...,2026-05-01 17:06:00,2026-05-01 18:21:00,75.0,LEON CUEVA LUIS ALBERTO,1,Si,2-110,Zamora,175857,OT [01] Cuadrilla Zamora 2026-05-01 (008) LL.pdf
386002,4,se_labora,"SE LABORA: LL, JCR de 17:06 a 18:21",LABORA,·,No,No,No,·,·,...,2026-05-01 00:00:01,2026-05-01 00:00:02,0.0,LEON CUEVA LUIS ALBERTO,1,Si,2-110,Zamora,175857,OT [01] Cuadrilla Zamora 2026-05-01 (008) LL.pdf


In [26]:
#filtered_df = df.query( f"Evento == Se labora F.A y L.V en horario de 08:00 a 13:00 y de 14:00 a 18:38" ).sort_values(by='Item')
filtered_df = df[df["Evento"].str.startswith("Se labora F.A y L.V en horario de 08:00 a 13:00 y de 14:00 a 18:38")].sort_values(by='Item')
filtered_df

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
169113,16,se_labora,Se labora F.A y L.V en horario de 08:00 a 13:0...,LABORA,·,No,No,No,·,·,...,2025-08-14 00:00:01,2025-08-14 00:00:02,0,AMBULUDI SILVA FAUSTO JOEL,1,Si,4-118,El Pangui - La Alfonsina - Los Bayanes - Abig ...,159195,OT [23] Agencia El Pangui 2025-08-14 (055) FA.pdf


### Volver a carga DeltaLake

In [16]:
dt = DeltaTable(table_path)
df = dt.to_pandas()
dt.version()


5313

#### Historia de Delta Lake

In [29]:
print("\n--- Table History ---")
history = dt.history(limit = 10)
for commit in history:
    print(
        f"Version: {commit['version']}, "
        f"Timestamp: {datetime.fromtimestamp(commit['timestamp']/1000)}, "
        f"Operation: {commit['operation']}"
    )



--- Table History ---
Version: 4185, Timestamp: 2026-02-24 15:14:16.833000, Operation: MERGE
Version: 4184, Timestamp: 2026-02-24 14:25:26.645000, Operation: MERGE
Version: 4183, Timestamp: 2026-02-24 14:23:47.639000, Operation: MERGE
Version: 4182, Timestamp: 2026-02-24 14:23:10.838000, Operation: WRITE
Version: 4181, Timestamp: 2026-02-24 10:49:05.109000, Operation: WRITE
Version: 4180, Timestamp: 2026-02-24 08:12:56.622000, Operation: MERGE
Version: 4179, Timestamp: 2026-02-24 08:11:35.811000, Operation: WRITE
Version: 4178, Timestamp: 2026-02-24 08:10:34.795000, Operation: WRITE
Version: 4177, Timestamp: 2026-02-24 08:09:48.785000, Operation: WRITE
Version: 4176, Timestamp: 2026-02-24 08:09:11.792000, Operation: WRITE


### Retaurar a una version anterior de Deltalake - Timetravel

In [ ]:
target_version = 2620

# --- Restore using a version number ---
dt.restore(target_version)
        
print(f"✅ Restauración exitosa! La tabla se encuentra en la version {dt.version()}.")

### Optimización y Aspirado

In [5]:
dt.optimize.compact()

{'numFilesAdded': 1,
 'numFilesRemoved': 610,
 'filesAdded': '{"avg":18783854.0,"max":18783854,"min":18783854,"totalFiles":1,"totalSize":18783854}',
 'filesRemoved': '{"avg":60373.1131147541,"max":31027684,"min":7544,"totalFiles":610,"totalSize":36827599}',
 'partitionsOptimized': 1,
 'numBatches': 1012,
 'totalConsideredFiles': 610,
 'totalFilesSkipped': 0,
 'preserveInsertionOrder': True}

In [6]:
dt.vacuum(retention_hours=100, enforce_retention_duration=False, dry_run=True)


['part-00000-5ad0d411-1b8c-470f-bed3-4bbbe701cb76-c000.snappy.parquet',
 'part-00000-dcf304f8-8153-4668-8bb0-40ea8c34c1a6-c000.snappy.parquet',
 'part-00000-4e59cd6f-3e90-45c2-a594-49ca905a0278-c000.snappy.parquet',
 'part-00000-2a667267-d951-43b7-b4ad-1a0687602d27-c000.snappy.parquet',
 'part-00000-d5886c40-0178-4693-a7d9-ebd66014ae0d-c000.snappy.parquet',
 'part-00000-5ac164bb-b29f-4f4d-a548-c0b78a400ea8-c000.snappy.parquet']

## Limpieza de dataset

In [ ]:
df

### Cambio de Tipo de Cuenta mal escrita

Hay casos en los que las Cuentas se guardan con espacios al final. por ejemplo `'CORRECTIVO '` y también `'CORRECTIAS '`
Se ha corregido este error a partir de la version. `gestion 5.1` sin embargo es necesario corregir los datos que ya 
constan en el dataset. El Siguiente código identifica y corrige estas variaciones para tener valores unificados del tipo
de actividad

In [8]:
"""
    El objetivo es identificar el tipo de actividad con un espacio al final 
    para ser reemplazadas por un mismo valor uniforme
    
    FILTROS:
    * Identificamos el texto: 'CORRECTIVAS '  "CORRECTIVO "
    
"""
mask = df['Tipo'] == "CORRECTIVO "
filtered_df = df[mask]
filtered_df


,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
3028,7,ALUMBRADO,La Gruta estructura 127962 se realiza revisió...,NO PROG,Bomboiza,No,No,No,CORRECTIVO,·,...,2025-10-23 14:55:00,2025-10-23 15:35:00,40.0,GUZMAN BARROS MARCO FERNANDO,1,No,4-52,"Gualaquiza, Aguacate, La Gruta.",163772,OT [09] Cuadrilla Gualaquiza 2025-10-23 (045) ...
3029,11,MEDIDORES,En Piuntza de Guadalupe se encuentra breaker ...,NO PROG,Yacuambi,No,No,No,CORRECTIVO,·,...,2025-10-21 17:10:00,2025-10-21 18:00:00,50.0,LOZANO SIGCHO NAUN ENRIQUE,1,Si,R-171,yacuambi - Piuntza de Guadalupe,163813,OT [03] Cuadrilla Yacuambi 2025-10-21 (017) NL...
3033,2,ACOMETIDAS,En el Sector La Fragancia se realiza inspecció...,PROG,Zamora I,No,No,No,CORRECTIVO,·,...,2025-10-23 09:00:00,2025-10-23 09:30:00,30.0,JARA NARVAEZ GALO SILVERIO,1,No,2-61,"ZAMORA, FRAGANCIA, RETORNO, LA ZURITA, RIO BLANCO",163778,OT [21] Agencia Zamora 2025-10-23 (050) GJ.pdf
3061,8,REDES,"Los Hachos, se revisa la derivación de Los Ha...",NO PROG,Yantzaza III,No,No,No,CORRECTIVO,·,...,2025-10-26 20:45:00,2025-10-26 20:55:00,10.0,ALEJANDRO PACHAR AGUSTIN EDUARDO,1,Si,2-112,Yantzaza y cantones aledaños.,163932,OT [04] Cuadrilla Yantzaza 2025-10-26 (022) AA...
47304,4,?,En Guaguayme se encuentra acometida arrancada ...,PROG,Los Encuentros,No,No,No,CORRECTIVO,·,...,2025-10-22 09:45:00,2025-10-22 11:00:00,75.0,LOZANO SIGCHO NAUN ENRIQUE,1,No,R-171,Yacuambi - Piuntza Alto de La Paz y Venecia,163895,OT [03] Cuadrilla Yacuambi 2025-10-22 (017) NL...
47310,5,ALUMBRADO,RECLAMO Nro. 1103803837 (21 - octubre 09:59). ...,PROG,Bomboiza,No,No,No,CORRECTIVO,·,...,2025-10-23 11:10:00,2025-10-23 12:40:00,90.0,MENDIETA MENDIETA HENRRY ALEXANDER,1,No,2-102,"Santa Cruz, El Quimi.",163753,OT [08] Cuadrilla El Pangui 2025-10-23 (042) H...
47344,1,ACOMETIDAS,En el Pangui DAÑO REPORTADO POR Centro de Con...,NO PROG,El Pangui,No,No,No,CORRECTIVO,·,...,2025-10-24 08:00:00,2025-10-24 08:30:00,30.0,AMBULUDI SILVA FAUSTO JOEL,1,No,4-118,El Pangui - Gualaquiza,163828,OT [23] Agencia El Pangui 2025-10-24 (058) FA.pdf
60234,4,REDES,En la estructura #502683 se realiza maniobras...,NO PROG,Yantzaza III,No,No,No,CORRECTIVO,·,...,2025-10-24 09:00:00,2025-10-24 10:00:00,60.0,AMARI ORDONEZ JUNIOR IVAN,1,No,R-180,"Yantzaza, Zumbi, Cuje, Cerro Santa Barbara",163879,OT [06] Cuadrilla Paquisha 2025-10-24 (034) JA...
86714,14,REDES,En Flor del Bosque por información de los mor...,NO PROG,Yacuambi,No,No,No,CORRECTIVO,·,...,2025-10-21 19:00:00,2025-10-21 20:00:00,60.0,LOZANO SIGCHO NAUN ENRIQUE,1,Si,R-171,yacuambi - Piuntza de Guadalupe,163813,OT [03] Cuadrilla Yacuambi 2025-10-21 (017) NL...
108976,14,REDES,Tsarums se revisa red que todo este normal y ...,NO PROG,Bomboiza,No,No,No,CORRECTIVO,·,...,2025-10-24 18:20:00,2025-10-24 19:00:00,40.0,GUZMAN BARROS MARCO FERNANDO,1,Si,4-52,"Gualaquiza - Bomboiza - Tunduli, Ahantas, Tsar...",163841,OT [09] Cuadrilla Gualaquiza 2025-10-24 (045) ...


In [9]:
print(f"Existen {mask.sum()} filas por modificar.")
df.loc[mask, 'Tipo'] = "CORRECTIVO"
print("✅ Se ha actualizado el Dataframe")

filtered_df = df[mask] # valores para actualizar en Deltalake

Existen 52 filas por modificar.
✅ Se ha actualizado el Dataframe


Lo mismo para las demas correcciones a realizar

In [10]:
mask = df['Tipo'] == "ACTGIS"
filtered_df = df[mask]
print(f"Existen {mask.sum()} filas por modificar.")
df.loc[mask, 'Tipo'] = "PREDICTIVO"
print("✅ Se ha actualizado el Dataframe")

filtered_df = df[mask] # valores para actualizar en Deltalake

Existen 0 filas por modificar.
✅ Se ha actualizado el Dataframe


### Identificar Cuentas de tipo "Servicios_Ocasionales"

In [11]:
"""
    El objetivo es identificar el tipo de actividad (Cuenta) que se encuentran como desconocidas (?) 
    pero que se pueden atribuir como cuenta de tipo "Servicios_Ocasionales
    
    FILTROS:
    * Debe contar con un Alimentador identificado
    * Cuenta: Desconocida (?)
    * Tipo: RUTINARIA
    * Que en el texto (Evento) contengan las palabras, 'acometida' o 'medidor'

    Estas filas son de Tipo = Servicios_Ocasionales, 

    primero generamos una máscara para identificarlas, luego aplicamos el cambio de cuenta,
    verificamos y guardamos en el Dataset.
"""
cond1 = df['Alimentador'] != "·"
cond2 = df['Tipo'].str.contains("RUTINARIA", regex=False, na=False, case=False)
cond3 = df['Cuenta'] == "?" 
cond4 = df['Evento'].str.contains("s/o", regex=False, na=False, case=False)

# Combine the conditions with the "&" (AND) operator to create the final boolean mask
mask = cond1 & cond2 & cond3 & cond4

# Apply the mask to the DataFrame to get the filtered result
filtered_df = df[mask]

# Display the filtered DataFrame
filtered_df


,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
3416,10,?,En el Pangui se instala un S/O en el estructur...,PROG,El Pangui,No,No,No,RUTINARIA,·,...,2026-01-27 15:00:00,2026-01-27 15:20:00,20.0,AMBULUDI SILVA FAUSTO JOEL,1,No,4-118,El Pangui - Uwents - Los Bayanes - San Roque,169627,OT [23] Agencia El Pangui 2026-01-27 (058) FA.pdf
16589,8,?,En Rancho Alegre se realiza inspección para S/...,PROG,Zamora II,No,No,No,RUTINARIA,·,...,2026-02-09 16:30:00,2026-02-09 16:45:00,15.0,JARA NARVAEZ GALO SILVERIO,1,No,2-61,"ZAMORA, SAN MARCOS, QUEBRADA DE CUMBARATZA, CU...",170562,OT [21] Agencia Zamora 2026-02-09 (050) GJ.pdf
47728,5,?,En Zamora parque Lineal se instala algunos S/O...,PROG,Zamora I,No,No,No,RUTINARIA,·,...,2025-11-07 11:25:00,2025-11-07 12:30:00,65.0,JARA NARVAEZ GALO SILVERIO,1,No,2-61,Zamora,164675,OT [21] Agencia Zamora 2025-11-07 (050) GJ.pdf
74443,4,?,En Yantzaza en calle Jorge Mosquera en estruc...,NO PROG,Yantzaza III,No,No,No,RUTINARIA,·,...,2025-10-15 09:40:00,2025-10-15 10:10:00,30.0,BARRAZUETA GONZAGA SERVIO GUILLERMO,4,No,2-112,Yantzaza- Chicaña,163229,OT [04] Cuadrilla Yantzaza 2025-10-15 (021) SB...
74854,5,?,En el Barrio 2 de Noviembre se instala 4 S/O 2...,PROG,Zamora I,No,No,No,RUTINARIA,·,...,2025-10-31 11:50:00,2025-10-31 13:30:00,100.0,JARA NARVAEZ GALO SILVERIO,1,No,2-61,ZAMORA,164308,OT [21] Agencia Zamora 2025-10-31 (050) GJ.pdf
74906,7,?,En el Barrio Santa Elena se retira S/O en la C...,PROG,Zamora I,No,No,No,RUTINARIA,·,...,2025-11-07 13:15:00,2025-11-07 13:30:00,15.0,JARA NARVAEZ GALO SILVERIO,1,No,2-61,Zamora,164675,OT [21] Agencia Zamora 2025-11-07 (050) GJ.pdf
109386,2,?,En el Centro Comercial se retira S/O 25 metros...,PROG,Zamora I,No,No,No,RUTINARIA,·,...,2025-10-31 09:30:00,2025-10-31 10:00:00,30.0,JARA NARVAEZ GALO SILVERIO,1,No,2-61,ZAMORA,164308,OT [21] Agencia Zamora 2025-10-31 (050) GJ.pdf
158358,5,?,En Rancho Alegre de Cumbaratza se Notifica a l...,PROG,Zamora II,No,No,No,RUTINARIA,·,...,2025-11-18 12:00:00,2025-11-18 12:15:00,15.0,JARA NARVAEZ GALO SILVERIO,1,No,2-61,"Zamora, Timbara, Rancho Alegre, Descanso",165280,OT [21] Agencia Zamora 2025-11-18 (050) GJ.pdf
166224,3,?,En San Roque se instala un S/O en el estructur...,PROG,Los Encuentros,No,No,No,RUTINARIA,·,...,2025-10-16 08:50:00,2025-10-16 09:30:00,40.0,AMBULUDI SILVA FAUSTO JOEL,1,No,4-118,El Pángui - Las Orquideas - Pachicutza - Pachkius,163310,OT [23] Agencia El Pangui 2025-10-16 (058) FA.pdf
179470,6,?,En Timbara se retira S/O 30 metros de TX6 estr...,PROG,Zamora II,No,No,No,RUTINARIA,·,...,2026-02-19 11:10:00,2026-02-19 11:40:00,30.0,JARA NARVAEZ GALO SILVERIO,1,No,2-61,"Zamora, Timbara",171111,OT [21] Agencia Zamora 2026-02-19 (050) GJ.pdf


In [12]:
print(f"Existen {mask.sum()} filas por modificar.")
df.loc[mask, 'Cuenta'] = "Servicios_Ocasionales"
print("✅ Se ha actualizado el Dataframe")

filtered_df = df[mask] # valores para actualizar en Deltalake

Existen 59 filas por modificar.
✅ Se ha actualizado el Dataframe


### Identificar Cuentas de tipo "MEDIDORES"

In [13]:
"""
    El objetivo es identificar el tipo de actividad (Cuenta) desconocidas (?) para separarlas de las que nos interesan (REDES)
    
    FILTROS:
    * Alimentador Zamora I
    * Tipo: Correctiva o Preventiva
    * Cuenta: Desconocida (?)
    * Que en el texto (Evento) contengan la palabra, 'medidor'

    Estas filas son de Tipo = MEDIDORES, 

    primero generamos una máscara para identificarlas, luego aplicamos el cambio de cuenta,
    verificamos y guardamos en el Dataset.
"""
mask = (
    (df['Alimentador'] != "·") &
    
    (
        df['Tipo'].str.contains("CORRECTIVO", regex=False, na=False, case=False) |
        df['Tipo'].str.contains("PREVENTIVO", regex=False, na=False, case=False)
    ) &
    
    (df['Cuenta'] == "?") &
    
    (
        df['Evento'].str.contains("medidor", regex=False, na=False, case=False)  # "acometida"
    )
)

# Apply the mask to the DataFrame to get the filtered view (optional)
filtered_df = df[mask]
filtered_df

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
1,10,?,En El Pangui se retira el medidor# 34911 trifá...,PROG,El Pangui,No,No,No,CORRECTIVO,·,...,2025-08-14 15:10:00,2025-08-14 16:30:00,80.0,AMBULUDI SILVA FAUSTO JOEL,1,No,4-118,El Pangui - La Alfonsina - Los Bayanes - Abig ...,159195,OT [23] Agencia El Pangui 2025-08-14 (058) FA.pdf
364740,7,?,"En La Recta, estructura. 72434 se revisa el me...",NO PROG,El Pangui,No,No,No,PREVENTIVO,·,...,2025-08-26 15:30:00,2025-08-26 16:00:00,30.0,MENDIETA MENDIETA HENRRY ALEXANDER,3,No,2-102,"El Porvenir, El Pangui.",159878,OT [08] Cuadrilla El Pangui 2025-08-26 (042) H...
368184,3,?,"En La Recta, estructura. 72439 se apoya al gru...",PROG,El Pangui,No,No,No,PREVENTIVO,·,...,2025-08-21 08:30:00,2025-08-21 12:45:00,255.0,MENDIETA MENDIETA HENRRY ALEXANDER,1,No,2-102,"La Palmira, Pakintza, Michanunka.",159583,OT [08] Cuadrilla El Pangui 2025-08-21 (042) H...
378551,13,?,"DAÑO REPORTADO POR Centro de Control, Nambija,...",NO PROG,Nambija,No,No,No,CORRECTIVO,·,...,2026-02-06 19:15:00,2026-02-06 20:10:00,55.0,RIOS RIOS FRANCISCO FERNANDO,3,Si,2-110,"Zamora, Cumbaratza, Nambija",170375,OT [01] Cuadrilla Zamora 2026-02-06 (006) FR.pdf
410279,3,?,"San Pedro, estructura #028531, arreglo acometi...",PROG,Bomboiza,No,No,No,CORRECTIVO,·,...,2026-04-28 08:55:00,2026-04-28 09:35:00,40.0,BUELE UYAGUARI CESAR CRISTIAN,0,No,2-107,"Gualaquiza, Roldos, Sevilla",175664,OT [24] Agencia Gualaquiza 2026-04-28 (061) CB...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
419975,6,?,"Condor, estructura #191443, restitucion medido...",NO PROG,Bomboiza,No,No,No,CORRECTIVO,·,...,2026-03-24 11:15:00,2026-03-24 12:20:00,65.0,BUELE UYAGUARI CESAR CRISTIAN,1,No,2-107,"Gualaquiza; Guayusal, Narvez, Osococha",173365,OT [24] Agencia Gualaquiza 2026-03-24 (061) CB...
419981,12,?,"Osococha, estructura #069968, restitucion medi...",PROG,Bomboiza,No,No,No,CORRECTIVO,·,...,2026-03-24 15:25:00,2026-03-24 16:30:00,65.0,BUELE UYAGUARI CESAR CRISTIAN,1,No,2-107,"Gualaquiza; Guayusal, Narvez, Osococha",173365,OT [24] Agencia Gualaquiza 2026-03-24 (061) CB...
420049,5,?,En el Pangui DAÑO REPORTADO POR Centro de Cont...,NO PROG,El Pangui,No,No,No,CORRECTIVO,·,...,2026-04-27 12:00:00,2026-04-27 12:20:00,20.0,AMBULUDI SILVA FAUSTO JOEL,1,No,4-118,El Pangui,175561,OT [23] Agencia El Pangui 2026-04-27 (058) FA.pdf
420072,9,?,"Gualaquiza, Sor Consuelo Iglesias y 16 de Agos...",PROG,Bomboiza,No,No,No,CORRECTIVO,·,...,2026-03-20 15:00:00,2026-03-20 16:20:00,80.0,BUELE UYAGUARI CESAR CRISTIAN,1,No,2-107,"Gualaquiza, Osococha, Kayamas",173122,OT [24] Agencia Gualaquiza 2026-03-20 (061) CB...


In [14]:
# Actualizamos los valores de Cuenta en el dataframe completo para luego guardarlo en el Datalake

# df.loc[mask, 'Cuenta'] = "ACOMETIDAS"
filtered_df.loc[mask,'Cuenta'] = "MEDIDORES"
filtered_df

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
1,10,MEDIDORES,En El Pangui se retira el medidor# 34911 trifá...,PROG,El Pangui,No,No,No,CORRECTIVO,·,...,2025-08-14 15:10:00,2025-08-14 16:30:00,80.0,AMBULUDI SILVA FAUSTO JOEL,1,No,4-118,El Pangui - La Alfonsina - Los Bayanes - Abig ...,159195,OT [23] Agencia El Pangui 2025-08-14 (058) FA.pdf
364740,7,MEDIDORES,"En La Recta, estructura. 72434 se revisa el me...",NO PROG,El Pangui,No,No,No,PREVENTIVO,·,...,2025-08-26 15:30:00,2025-08-26 16:00:00,30.0,MENDIETA MENDIETA HENRRY ALEXANDER,3,No,2-102,"El Porvenir, El Pangui.",159878,OT [08] Cuadrilla El Pangui 2025-08-26 (042) H...
368184,3,MEDIDORES,"En La Recta, estructura. 72439 se apoya al gru...",PROG,El Pangui,No,No,No,PREVENTIVO,·,...,2025-08-21 08:30:00,2025-08-21 12:45:00,255.0,MENDIETA MENDIETA HENRRY ALEXANDER,1,No,2-102,"La Palmira, Pakintza, Michanunka.",159583,OT [08] Cuadrilla El Pangui 2025-08-21 (042) H...
378551,13,MEDIDORES,"DAÑO REPORTADO POR Centro de Control, Nambija,...",NO PROG,Nambija,No,No,No,CORRECTIVO,·,...,2026-02-06 19:15:00,2026-02-06 20:10:00,55.0,RIOS RIOS FRANCISCO FERNANDO,3,Si,2-110,"Zamora, Cumbaratza, Nambija",170375,OT [01] Cuadrilla Zamora 2026-02-06 (006) FR.pdf
410279,3,MEDIDORES,"San Pedro, estructura #028531, arreglo acometi...",PROG,Bomboiza,No,No,No,CORRECTIVO,·,...,2026-04-28 08:55:00,2026-04-28 09:35:00,40.0,BUELE UYAGUARI CESAR CRISTIAN,0,No,2-107,"Gualaquiza, Roldos, Sevilla",175664,OT [24] Agencia Gualaquiza 2026-04-28 (061) CB...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
419975,6,MEDIDORES,"Condor, estructura #191443, restitucion medido...",NO PROG,Bomboiza,No,No,No,CORRECTIVO,·,...,2026-03-24 11:15:00,2026-03-24 12:20:00,65.0,BUELE UYAGUARI CESAR CRISTIAN,1,No,2-107,"Gualaquiza; Guayusal, Narvez, Osococha",173365,OT [24] Agencia Gualaquiza 2026-03-24 (061) CB...
419981,12,MEDIDORES,"Osococha, estructura #069968, restitucion medi...",PROG,Bomboiza,No,No,No,CORRECTIVO,·,...,2026-03-24 15:25:00,2026-03-24 16:30:00,65.0,BUELE UYAGUARI CESAR CRISTIAN,1,No,2-107,"Gualaquiza; Guayusal, Narvez, Osococha",173365,OT [24] Agencia Gualaquiza 2026-03-24 (061) CB...
420049,5,MEDIDORES,En el Pangui DAÑO REPORTADO POR Centro de Cont...,NO PROG,El Pangui,No,No,No,CORRECTIVO,·,...,2026-04-27 12:00:00,2026-04-27 12:20:00,20.0,AMBULUDI SILVA FAUSTO JOEL,1,No,4-118,El Pangui,175561,OT [23] Agencia El Pangui 2026-04-27 (058) FA.pdf
420072,9,MEDIDORES,"Gualaquiza, Sor Consuelo Iglesias y 16 de Agos...",PROG,Bomboiza,No,No,No,CORRECTIVO,·,...,2026-03-20 15:00:00,2026-03-20 16:20:00,80.0,BUELE UYAGUARI CESAR CRISTIAN,1,No,2-107,"Gualaquiza, Osococha, Kayamas",173122,OT [24] Agencia Gualaquiza 2026-03-20 (061) CB...


### Guardar cambios en Deltalake

In [ ]:
import pyarrow as pa 

edited_data = pa.Table.from_pandas( filtered_df )

try:

    unique_key_predicate = "target.id_ot = source.id_ot AND target.Item = source.Item"

    (dt.merge(
                    source=edited_data,
                    predicate=unique_key_predicate,
                    source_alias="source",
                    target_alias="target"
                )
                .when_matched_update_all()  # Rule 1: If an activity exists, update it.
                .when_not_matched_insert_all()  # Rule 2: If it's a new activity, insert it.
                .execute()
    )
    print("✅ **Successfully saved changes to Delta Lake!**")
except Exception as e:
    print(f"❌ **Error saving to Delta Lake:** {e}")

dt = DeltaTable(table_path)
df = dt.to_pandas()
dt.version()


✅ **Successfully saved changes to Delta Lake!**


## Mantenimiento `matriz_actividades`

### Limpieza de InicioEvento y FinEvento

In [17]:
# 1. Elimina espacios en blanco del Inicio y del Fin del evento

df['InicioEvento'] = df['InicioEvento'].str.strip()
df['FinEvento'] = df['FinEvento'].str.strip()

In [18]:
# Crear una mascara para los elementos que tienen 20 caracteres
# Create mask - True for rows that DON'T match the pattern
mask = ~df['FinEvento'].astype(str).str.match(r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$')
# Handle NaN values explicitly
mask = mask | df['FinEvento'].isna()

In [19]:
df[mask].tail(5)

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
385526,1,informativa,FERIADO SEMANA SANTA\nESTE DIA NO SE PRESENTO ...,INFO,·,No,No,No,·,·,...,None,None,NaN,QUIROGA ORDONEZ CARLOS HERNAN,1,No,4-33,Guayzimi,174036,OT [07] Cuadrilla Guayzimi 2026-04-03 (038) CQ...
393102,3,se_labora,"SE LABORA CQ, JL de:...",LABORA,·,No,No,No,·,·,...,2026-04-30T00:00:00-05:00 00:00:01,2026-04-30T00:00:00-05:00 00:00:02,0.0,QUIROGA ORDONEZ CARLOS HERNAN,1,Si,4-33,Guayzimi,175860,OT [07] Cuadrilla Guayzimi 2026-04-30 (038) CQ...
408754,1,informativa,"DIA FESTIVO ""DIA DEL TRABAJO""\rNo se presentan...",INFO,·,No,No,No,·,·,...,2026-05-01T00:00:00-05:00 00:00:01,2026-05-01T00:00:00-05:00 00:00:02,0.0,AMARI ORDONEZ JUNIOR IVAN,1,Si,R-184,Paquisha,175883,OT [06] Cuadrilla Paquisha 2026-05-01 (034) JA...
413018,2,transporte,Traslado desde la agencia de la EERSSA Zamora ...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2026-02-05 14:30:00,2026-02-05 15:50:00,80.0,MORALES RIVERA LUIS ALBERTO,3,No,2-91,"Zamora, San Carlos de las Minas",170344,OT [02] Alumbrado Zamora 2026-02-05 (012) LM.pdf
416574,1,informativa,(DIA FESTIVO): '' Decreto Ejecutivo 172 ''\rEn...,INFO,·,No,No,No,·,·,...,2025-10-10T00:00:00-05:00 00:00:01,2025-10-10T00:00:00-05:00 00:00:02,0.0,VERA SANCHEZ LIVIO STALIN,1,Si,2-102,El Pangui.,162995,OT [08] Cuadrilla El Pangui 2025-10-10 (043) L...


In [ ]:
# Procesar estas fechas y horas para que todas tengan el mismo formato:

# 2. Funcion para eliminar el componente de Time Zone
#    éste es introducido cuando en actividades no se consigue una 'fecha_moda' 
#    y es necesario utilizar la 'fecha' de hoja_uno.   

def elimina_timezone( fecha ):
  try:
    
    # Se elimina el componente de Zona Horaria
    fecha_inicio = fecha.replace('T', ' ').split()
    fecha_inicio = fecha_inicio[0]+' '+fecha_inicio[-1]
    print(f"[OK] SI Se convierte la cadena de caracteres: {fecha}")
    return fecha_inicio
    
  
  except:
    print(f"[X] No fue posible convertir la cadena de caracteres:  {fecha}")
    return fecha

df.loc[mask, 'InicioEvento'] = df.loc[mask, 'InicioEvento'].apply( lambda x: elimina_timezone(x))
df.loc[mask, 'FinEvento'] = df.loc[mask, 'FinEvento'].apply( lambda x: elimina_timezone(x))


[X] No fue posible convertir la cadena de caracteres:  None
[X] No fue posible convertir la cadena de caracteres:  None
[X] No fue posible convertir la cadena de caracteres:  None
[X] No fue posible convertir la cadena de caracteres:  None


In [21]:
def calcular_minutos_transcurridos( start_times, end_times ):
  """
  This function works because subtracting two pandas Series of datetimes
  is a vectorized operation.
  """
  try:
    # Ensure columns are in datetime format first
    start_times = pd.to_datetime(start_times)
    end_times = pd.to_datetime(end_times)

    time_difference = end_times - start_times
    # Return the difference in minutes
    return (time_difference.dt.total_seconds() / 60).astype(int)
  except Exception as e:
    print(f" EXCEPTION:\n{e}")

In [ ]:
#  ANALIZAR UNA OT filtrando por su indice

filtered_df = df.query( f"id_ot == 160106" ).sort_values(by='Item')
filtered_df

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
394358,1,informativa,En la agencia de la EERSSA se coordina los tra...,PROG,·,No,No,No,RUTINARIA,·,...,2025-08-29 07:30:00,2025-08-29 07:40:00,10,MORALES RIVERA LUIS ALBERTO,2,Si,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf
394359,2,transporte,Traslado desde la agencia de la EERSSA Zamora ...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2025-08-29 07:40:00,2025-08-29 07:50:00,10,MORALES RIVERA LUIS ALBERTO,2,Si,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf
394360,3,REDES,En el sector de la Fragancia se realiza lo sig...,PROG,Zamora I,No,No,No,CORRECTIVO,·,...,2025-08-29 07:50:00,2025-08-29 12:55:00,305,MORALES RIVERA LUIS ALBERTO,2,Si,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf
394361,4,lunch,Lunch en La Fragancia.,ALIMEN,·,No,No,No,LUNCH,·,...,2025-08-29 12:55:00,2025-08-29 13:55:00,60,MORALES RIVERA LUIS ALBERTO,2,No,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf
394362,5,ALUMBRADO,Sector de la Fragancia se continua con la repa...,PROG,Zamora I,No,No,No,CORRECTIVO,·,...,2025-08-29 13:55:00,2025-08-29 19:29:00,334,MORALES RIVERA LUIS ALBERTO,2,Si,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf
394363,7,se_labora,SE LABORA: LM y LL de 07:30 a 12:55 y de 13:55...,LABORA,·,No,No,No,·,·,...,2025-08-29 00:00:01,2025-08-29 00:00:02,0,MORALES RIVERA LUIS ALBERTO,2,Si,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf
394364,8,se_labora,SE LABORA: RY se encuentra con reposo médico.\...,LABORA,·,No,No,No,·,·,...,2025-08-29 00:00:01,2025-08-29 00:00:02,0,MORALES RIVERA LUIS ALBERTO,2,Si,2-110,La Fragancia,160106,OT [02] Alumbrado Zamora 2025-08-29 (012) LM.pdf


In [22]:
#filtered_df.loc[93810, 'InicioEvento'] = '2021-04-23 22:10:00'
filtered_df['Duracion'] = calcular_minutos_transcurridos(
    filtered_df['InicioEvento'],
    filtered_df['FinEvento']
)

#### Para corregir la Fecha final cuando se coloca "00:00:00" en lugar de "23:59:00" al finalizar el día

In [23]:
# Mascara para determinar duracion menor a 0, para volver a calcular las horas. 

mask = ( df['Duracion'] < 0 )

In [24]:
# Crear una mascara para aplicar los cambios

# Create mask for rows that end with '00:00:00'
mask = df['Duracion'].astype(str).str.endswith('00:00:00') & df['FinEvento'].notna() & ( df['Duracion'] < -1300 )

In [25]:
# Muestra cuantos casos se ha identificado

true_indices = mask[mask].index
len(true_indices.tolist())

0

### Vuelve a calcular la columna 'Duracion' en minutos

In [ ]:
# Ejecuta el reemplazo de las horas

#df.loc[mask, 'FinEvento'] = df.loc[mask, 'FinEvento'].astype(str).str[:-8] + '23:59:00'
df['Duracion'] = calcular_minutos_transcurridos(
    df['InicioEvento'],
    df['FinEvento']
)

### Muestra fechas de actividades con posible conflicto

In [26]:
# filtered_df = df.query("FinEvento.str.endswith('00:00:00') and Alimentador != '·' ")
# filtered_df = df.query("Duracion < 0")

filtered_df = df.query("Duracion < 0")
filtered_df

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
134348,3,lunch,Lunch en La Y del Guismi.,ALIMEN,·,No,No,No,LUNCH,·,...,2026-01-29 12:00:00,2026-01-29 00:00:00,-720.0,MORALES RIVERA LUIS ALBERTO,3,No,2-91,"El Pangui, Gualaquiza",169771,OT [02] Alumbrado Zamora 2026-01-29 (012) LM.pdf
154428,2,ALUMBRADO,"R.-Pangui,1lum150w,apagada,pst 127974 se cambi...",PROG,El Pangui,No,No,No,CORRECTIVO,·,...,2026-01-22 19:45:00,2026-01-22 10:30:00,-555.0,ORELLANA BRAVO JORGE LUIS,3,No,2-30,EL PANGUI,169361,OT [no] Cuadrilla Loja 2026-01-22 (0) JOB.pdf
245662,5,transporte,"DAÑO REPORTADO POR CENTRO DE CONTROL, mediant...",TRANSP,·,No,No,No,TRANSPORTE,·,...,2026-02-14 19:30:00,2026-02-14 10:10:00,-560.0,CHAMBA CANGO PEDRO ROSALINO,1,Si,2-112,"YANTZAZA, PINCHO.",170886,OT [22] Agencia Yantzaza 2026-02-14 (055) PCH.pdf
248744,12,transporte,Traslado desde el sector de La Quebrada de Cum...,TRANSP,·,No,No,No,TRANSPORTE,·,...,2026-01-08 15:40:00,2026-01-08 11:56:00,-224.0,MORALES RIVERA LUIS ALBERTO,2,No,2-91,Zamora,168456,OT [02] Alumbrado Zamora 2026-01-08 (012) LM.pdf
303034,9,MEDIDORES,En Jembuentza se revisa medidor por no registr...,PROG,Yacuambi,No,No,No,PREVENTIVO,·,...,2026-01-30 13:30:00,2026-01-30 13:20:00,-10.0,JARA NARVAEZ GALO SILVERIO,1,No,2-61,"Yacuambi, Zamora.",169876,OT [21] Agencia Zamora 2026-01-30 (050) GJ.pdf
319399,3,MEDIDORES,"Portón estructura 128404 se revisa medidor, se...",NO PROG,Bomboiza,No,No,No,CORRECTIVO,·,...,2026-02-05 09:52:00,2026-02-05 08:52:00,-60.0,GUZMAN BARROS MARCO FERNANDO,1,No,2-111,"Gualaquiza, San José de Piunts, La Esperanza.",170310,OT [09] Cuadrilla Gualaquiza 2026-02-05 (045) ...
331269,14,informativa,"Cambio de acometida, medidor# 1000401129 estru...",PROG,·,No,No,No,CORRECTIVO,·,...,2026-03-03 15:20:00,2026-03-03 15:00:00,-20.0,CARTUCHE SILVA GERARDO PATRICIO,1,No,2-107,"Gualaquiza,Sab Miguel, El Ideal, Tiwintza, Zap...",171947,OT [24] Agencia Gualaquiza 2026-03-03 (062) GC...
342539,8,transporte,Desde Playas de Chivatos se retorna a Yacuambi,TRANSP,·,No,No,No,TRANSPORTE,·,...,2026-04-16 17:50:00,2026-04-16 17:23:00,-27.0,LOZANO SIGCHO NAUN ENRIQUE,2,No,4-52,Yacuambi - Zamora,174931,OT [03] Cuadrilla Yacuambi 2026-04-16 (017) NL...
346632,8,transporte,Nos trasladamos de Zumbi a Yantzaza,TRANSP,·,No,No,No,TRANSPORTE,·,...,2026-03-10 12:30:00,2026-03-10 12:15:00,-15.0,CHAMBA CANGO PEDRO ROSALINO,2,No,4-100,"Yantzaza, Zumbi",172419,OT [22] Agencia Yantzaza 2026-03-10 (055) PCH.pdf
346759,11,ALUMBRADO,"INSPECCIÓN, atendido, se cambia lámpara, de 10...",PROG,El Pangui,No,No,No,CORRECTIVO,·,...,2026-02-12 12:35:00,2026-02-12 12:00:00,-35.0,MORALES RIVERA LUIS ALBERTO,3,No,2-91,El Pangui,170802,OT [02] Alumbrado Zamora 2026-02-12 (012) LM.pdf


In [ ]:
df.query(f"Alimentador != '·'").sort_values(by="Duracion",ascending=True).head(10)

,Item,Cuenta,Evento,Actividad,Alimentador,Primario,Desconexion,SIG,Tipo,Materiales,...,InicioEvento,FinEvento,Duracion,Responsable,Colaboradores,HorasExtra,Vehiculo,Sitio,id_ot,Archivo
94962,1,REDES,DEL CENTRO DE CONTROL INFORMAN QUE EN LOS SECT...,INFO,Y DE,No,No,No,·,·,...,2020-01-01 00:00:01,2020-01-01 00:00:02,0,LITUMA CORDOVA CESAR RUBEN,1,Si,R-12,GUALAQUIZA.GUAYUZAL,40977,OT [24] Agencia Gualaquiza 2020-01-01 (043) CL...
163532,1,REDES,"CALL CENTER LOJA INFORMA QUE EN CHUCHUMBLETZA,...",INFO,RAN,No,No,No,·,·,...,2018-11-10 00:00:01,2018-11-10 00:00:02,0,AMARI ORDONEZ JUNIOR IVAN,1,Si,R-96,"El Pangui, Chuchumbletza, El Pincho",19165,OT [08] Cuadrilla El Pangui 2018-11-10 (031) J...
163675,11,MEDIDORES,"En el Domicilio del Sr. Oclides Sosa, medidor ...",PROG,Yantzaza III,No,No,No,CORRECTIVO,·,...,2021-10-27 14:20:00,2021-10-27 14:20:00,0,CHAMBA CANGO PEDRO ROSALINO,1,No,4-36,"Yantzaza, Panguintza",75209,OT [22] Agencia Yantzaza 2021-10-27 (052) PCH.pdf
389212,9,?,Del Centro de Control reportan S/S en el barr...,NO PROG,El Pangui,No,No,No,PREDICTIVO,·,...,2020-10-28 10:50:00,2020-10-28 10:50:00,0,CUENCA MURILLO JORGE VICENTE,2,No,,"El Pangui, El Guismi",55253,OT [23] Agencia El Pangui 2020-10-28 (0) JCM.pdf
339228,1,ALUMBRADO,Paquisha en la estructura. No. 29560 se arregl...,PROG,Paquisha,No,No,No,CORRECTIVO,·,...,2023-11-28 08:00:00,2023-11-28 08:00:00,0,MORALES RIVERA LUIS ALBERTO,3,No,2-91,"Paquisha, Santa Rosa, Nuevo Quito, Mayaycu, La...",119442,OT [02] Alumbrado Zamora 2023-11-28 (012) LM.pdf
94633,1,?,DE CALL CENTER LOJA INFORMAN QUE EN EL SECTOR...,INFO,P,No,No,No,·,·,...,2018-12-08 00:00:01,2018-12-08 00:00:02,0,LITUMA CORDOVA CESAR RUBEN,1,Si,R-101,LA MISIÓN DE BOMBOIZA,20536,OT [24] Agencia Gualaquiza 2018-12-08 (043) CL...
322635,1,?,"POR DISPOSICIÓN DE LA EERSSA, SE LABORA EN UNA...",INFO,Gualaquiza,No,No,No,·,·,...,2019-12-24 00:00:01,2019-12-24 00:00:02,0,RIOS RIOS FRANCISCO FERNANDO,0,Si,R-18,"Gualaquiza, Zamora",40645,OT [01] Cuadrilla Zamora 2019-12-24 (006) FR.pdf
360624,18,ALUMBRADO,INC. Nro. 1103745315. Gualaquiza barrio San Fr...,PROG,Bomboiza,No,No,No,CORRECTIVO,·,...,2025-05-27 16:55:00,2025-05-27 16:55:00,0,MORALES RIVERA LUIS ALBERTO,3,No,2-91,Gualaquiza,153869,OT [02] Alumbrado Zamora 2025-05-27 (012) LM.pdf
76878,21,se_labora,SE LABORA: HM; VC de 20:04 a 21:07.,LABORA,El Pangui 2,No,No,No,·,·,...,2023-12-15 00:00:01,2023-12-15 00:00:02,0,MENDIETA MENDIETA HENRRY ALEXANDER,2,Si,2-102,"Chayazapa, Las Orquídeas, Shaime, Tsarunts.",120495,OT [08] Cuadrilla El Pangui 2023-12-15 (039) H...
294473,1,?,CALL CENTER LOJA INFORMA QUE EN EL BARRIO 8 DE...,INFO,A,No,No,No,·,·,...,2019-03-23 00:00:01,2019-03-23 00:00:02,0,AMARI ORDONEZ JUNIOR IVAN,1,Si,R-96,8 de Diciembre,25863,OT [08] Cuadrilla El Pangui 2019-03-23 (031) J...


In [ ]:
df

# LEGACY

> Analisis posterio entre MongoDB y DeltaLake

## Conectar con DASK Local Cluster

In [11]:
# DASK
from dask.distributed import LocalCluster, as_completed
dask = LocalCluster().get_client()
dask.dashboard_link

INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 41201 instead
  warnings.warn(
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:45905
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:41201/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:39789'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:43903'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:45879'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:39097'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:3979

'http://127.0.0.1:41201/status'

## Recargar Librerias Dinámicamente


In [6]:
## RECARGAR LAS LIBRERIAS DINAMICAMENTE
from importlib import reload
from eerssa import gestionOT as ClaseOT
from eerssa import procesarOt as OrdenTrabajo             # Convert from PDF_ot to obj_ot
from eerssa import generarMatrizActividades as Actividades     # process ot.data["actividades"]
from eerssa import procesarActividades as ActividadesV30

In [13]:
reload( OrdenTrabajo )
reload( Actividades  )
reload( ClaseOT )
reload( ActividadesV30)

<module 'eerssa.procesarActividades' from '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/eerssa/procesarActividades.py'>

## Verificacion de OT desde MongoDB hacia DeltaLake

### Descargar y procesar [01] desde MongoDB 

In [ ]:
id_descarga = 97355

try:
  ot_test = CurrentCollection.find_one({'id_ot':id_descarga})
  if not ot_test:
    print(f" [X] No se pudo descargar la OT")
  else:
    activ = ActividadesV30.ConvertirOT_a_ActividadesCSV( ClaseOT.GestionOt.from_v30( ot_test ) )

except Exception as e:
  print(f"{e}")

In [35]:
import pyarrow as pa 

edited_data = pa.Table.from_pandas( activ )

try:

    unique_key_predicate = "target.id_ot = source.id_ot AND target.Item = source.Item"

    (dt.merge(
                    source=edited_data,
                    predicate=unique_key_predicate,
                    source_alias="source",
                    target_alias="target"
                )
                .when_matched_update_all()  # Rule 1: If an activity exists, update it.
                .when_not_matched_insert_all()  # Rule 2: If it's a new activity, insert it.
                .when_not_matched_by_source_delete(  # Rule 3: If an old activity is now gone...
                    predicate=f"target.id_ot = {id_descarga}"  # ...delete it, but only for the current OT.
                )
                .execute()
    )
    print("✅ **Successfully saved changes to Delta Lake!**")
except Exception as e:
    print(f"❌ **Error saving to Delta Lake:** {e}")

✅ **Successfully saved changes to Delta Lake!**


### Cursor para obtener todos los "id_ot" desde MongoDB

In [14]:
""" 
   OBTENER TODOS LOS 'id_ot' desde MongoDB
"""

try:
    # 1. Use a projection to only retrieve the 'id_ot' field.
    #    - {'id_ot': 1} means "include this field".
    #    - {'_id': 0} means "exclude the default _id field".
    cursor = CurrentCollection.find({}, {'id_ot': 1, '_id': 0})

    # 2. Create a list from the cursor results using a list comprehension.
    #    This iterates through each document in the cursor and extracts 'id_ot'.
    id_ot_list = [doc['id_ot'] for doc in cursor]

    # 3. Now you have your list of all 'id_ot' values.
    print(f"Successfully retrieved {len(id_ot_list)} 'id_ot' values.")
    if id_ot_list:
        print("First 10 values:", id_ot_list[:10])

except Exception as e:
    print(f"An error occurred: {e}")


An error occurred: name 'CurrentCollection' is not defined


In [15]:
"""
   Obtener todos los 'id_ot' existentes en DeltaLake
"""

delta_ids = df["id_ot"].unique()
len(delta_ids)

2203

In [ ]:
"""
   Difentecia de las ot que faltan en DeltaLake
"""
set_mongo = set(id_ot_list)
set_delta = set(delta_ids)

# Find which items in set_delta are not in set_mongo
new_ids_set = set_mongo.difference(set_delta)

# Convert the result back to a list
new_ids_to_process = list(new_ids_set)

print(f"Found {len(new_ids_to_process)} new IDs to be processed.")
# We sort the list here just for a predictable, clean output
print(f"New IDs: {sorted(new_ids_to_process)}")

In [ ]:
"""
   Descargar y procesar las OT faltantes y añadirlas al Delta Lake
"""
new_data_frames = []
for ot in new_ids_to_process:
  json_ot = CurrentCollection.find_one({"id_ot": ot})
  if not json_ot:
    print(f"No se pudo encontrar la OT con id_ot '{ot}' en MongoDB. Saltando.")
    continue
                
  obj_ot = OrdenTrabajo.GestionOt.from_dict(json_ot)
  new_data_frames.append(Actividades.ConvertirOT_a_ActividadesCSV(obj_ot))

In [ ]:
try:
  new_df = pd.concat(new_data_frames, ignore_index=True)
  write_deltalake(table_path, new_df, mode='append')
  print(f" [ EXITO ] DELTA LAKE Se han añadido {len(new_df)} filas a la tabla Delta en '{table_path}'.")
except Exception as e:
  print(f"Fallo al escribir en la tabla Delta: {e}")

 [ EXITO ] DELTA LAKE Se han añadido 121556 filas a la tabla Delta en './test/deltalake_2025'.


## FULL MONGO DOWNLOAD

Generar un nuevo archivo Delta Lake para unificar versiones - Ejecutado JULIO 2025

### Descarga de OT's desde MongoDB hacia Pickle y Delta Lake 

In [ ]:
import re
# Assuming CurrentCollection is a valid pymongo.collection.Collection object
# and is already connected to your database as in consumer.py.

# ... setup client, db, collection
cursor = CurrentCollection.find()
all_documents = cursor.to_list() 
# or simply: all_documents = list(cursor)
print(f"Loaded {len(all_documents)} documents into a list.")
client.close()



Loaded 21879 documents into a list.


In [ ]:
obj_list = []
for document in all_documents:
  ot = OrdenTrabajo.GestionOt.from_dict( document )
  obj_list.append( Actividades.ConvertirOT_a_ActividadesCSV(ot) )
df = pd.concat(obj_list, ignore_index=True)


In [ ]:
df["Fecha"] = pd.to_datetime(df["Fecha"])
df["InicioEvento"] = pd.to_datetime(df["InicioEvento"], format='mixed')
df["FinEvento"] = pd.to_datetime(df["FinEvento"], format='mixed')

df.to_pickle("/home/vlad/Documents/mongodb_v23.pkl")

/tmp/ipykernel_316235/3892611823.py:2: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df["InicioEvento"] = pd.to_datetime(df["InicioEvento"], format='mixed')
/tmp/ipykernel_316235/3892611823.py:3: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df["FinEvento"] = pd.to_datetime(df["FinEvento"], format='mixed')


In [ ]:
write_deltalake("/home/vlad/delta_v23", df)

In [ ]:
dt = DeltaTable("/home/vlad/delta_v23")

## Descargar OT faltantes desde MongoDB hacia DeltaLake

## Cargar Pickle para analisis

In [43]:
# DELTA LAKE Connection
# Verify the existence of the DELTA LAKE table
import pandas as pd
import numpy as np
from deltalake import DeltaTable
from datetime import datetime
from pprint import pprint


def borra_time_zone( fecha:str ):
  """
  Esta función elimina el componenete de Time Zone y deja solamente la fecha y hora. 
  En caso de que no contenga este componente deja el String intacto. 
  """
  fecha_inicio = fecha.replace('T', ' ').split()
  fecha_inicio = fecha_inicio[0]+' '+fecha_inicio[-1]
  return fecha_inicio

if not DeltaTable.is_deltatable(table_path):
    print(
        f"No se ha encontrado la base de datos PARQUET-DELTALAKE en la direccion:\n NO_DELTA_LAKE : {table_path}" )
else:
    dt = DeltaTable(table_path)
    df = dt.to_pandas()
    print(f"Conectado a la tabla Delta Lake en: {table_path}")

df.info()

Conectado a la tabla Delta Lake en: /home/vlad/delta_V30
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31308 entries, 0 to 31307
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Item           31308 non-null  int64 
 1   Cuenta         31308 non-null  object
 2   Evento         31308 non-null  object
 3   Actividad      31308 non-null  object
 4   Alimentador    31308 non-null  object
 5   Primario       31308 non-null  object
 6   Desconexion    31308 non-null  object
 7   SIG            31308 non-null  object
 8   Tipo           31308 non-null  object
 9   Materiales     31308 non-null  object
 10  Cuadrilla      31308 non-null  object
 11  Dia            31308 non-null  object
 12  Fecha          31308 non-null  object
 13  InicioEvento   31308 non-null  object
 14  FinEvento      31308 non-null  object
 15  Duracion       31308 non-null  int64 
 16  Responsable    31308 non-null  object
 17  Colaboradore

In [31]:
variant = df.copy()

In [44]:
df['InicioEvento'] = df['InicioEvento'].apply( lambda x: borra_time_zone(x))
df['FinEvento'] = df['FinEvento'].apply( lambda x: borra_time_zone(x))

In [34]:
dt.version()

147

In [ ]:
# Perform the merge operation
print("\n--- Merging changes back into Delta Table ---")

(
    dt.merge(
        source=df,
        predicate="target.id = source.id",
        source_alias="source",
        target_alias="target"
    )
    .when_matched_update_all()  # If id matches, update the row
    .when_not_matched_insert_all()  # If a new id is in the source, insert it
    .execute()
)

print("Merge complete.")

### ¿Son todas los items en 'Fecha' validos?

In [45]:
def is_valid_utc_format(text_input):
    """
    Checks if a string can be converted to a timezone-aware datetime.

    The function tests against a specific ISO 8601 format that includes a
    UTC offset, like "2024-05-31T00:00:00-05:00".

    Args:
        text_input: The string or value to check.

    Returns:
        - True: if the input is a string and matches the format.
        - False: if the input is not a string or does not match the format.
        - pd.NaT: if the input is a null-like value (e.g., None, np.nan).
    """
    # 1. Handle null-like inputs first
    if pd.isna(text_input):
        return pd.NaT

    # 2. Ensure the input is a string before attempting to parse
    if not isinstance(text_input, str):
        return False

    # 3. Try to parse the string using the specific format
    try:
        # The format string matches the user's example.
        # %Y: 4-digit year
        # %m: 2-digit month
        # %d: 2-digit day
        # T: Literal 'T' separator
        # %H:%M:%S: Hour, minute, second
        # %z: UTC offset (e.g., -0500). Pandas extends this to handle
        #     the colon format (-05:00) as well.
        # errors='raise' ensures that any parsing failure raises an exception.
        pd.to_datetime(text_input, format="%Y-%m-%d %H:%M:%S", errors='raise')
        return True
    except ValueError:
        # This exception is raised if the string does not match the format.
        return False



In [47]:
df['InicioEvento'][0]

'2024-07-17 08:00:00'

In [37]:
fechas_validas = variant['InicioEvento'].apply( lambda x: is_valid_utc_format(x) )
fechas_validas.unique()

array([ True, False])

In [39]:
false_indices = np.where(~fechas_validas)[0]
len(false_indices)

18151

In [41]:
last_wrong_date = false_indices[-1]

In [ ]:
# 2023-02-20T00:00:00-05:00 00:00:01

In [42]:
variant.iloc[last_wrong_date]

Item                                                             4
Cuenta                                                   se_labora
Evento           SE LABORA: RM, RY\nNo se presentan novedades e...
Actividad                                                   LABORA
Alimentador                                                      ·
Primario                                                        No
Desconexion                                                     No
SIG                                                             No
Tipo                                                             ·
Materiales                                                       ·
Cuadrilla                                         Zamora (Agencia)
Dia                                                        viernes
Fecha                                    2022-02-11T00:00:00-05:00
InicioEvento                    2022-02-11T00:00:00-05:00 00:00:01
FinEvento                       2022-02-11T00:00:00-05:00 00:0

In [ ]:
import datetime
# 2. Define a function to safely get the date
def safe_to_date(value):
    # Check if the value is a Timestamp or datetime object
    if isinstance(value, (pd.Timestamp, datetime.datetime)):
        return value.date()
    # If it's already a date object, just return it
    elif isinstance(value, datetime.date):
        return value
    # For any other type, return NaT (Not a Time)
    else:
        return pd.NaT

In [ ]:
df['dates_equal_Inicio'] = (df['Fecha'].dt.date == df['InicioEvento'].apply(safe_to_date))
df['dates_equal_Fin'] = (df['Fecha'].dt.date == df['FinEvento'].apply(safe_to_date))

In [ ]:
df['dates_equal_Inicio'].unique()

array([ True, False])

In [ ]:
falla_inicio = df.query("dates_equal_Inicio == False")
#falla_inicio[["Item","Responsable","id_ot","Fecha","InicioEvento","FinEvento"]]
falla_inicio["id_ot"].unique()

array([155505, 155762, 145572, 144647, 148359, 147423, 148549, 148277,
       148246, 146584, 147455, 147804, 148333, 149748, 149726, 149242,
       150570, 137497, 139625, 138768, 142840, 139823, 139690, 134776,
       137787, 128220, 139615, 144309, 141353, 144417, 134989, 143002,
       140128, 134479, 137253, 125688, 110252, 105181, 114346, 119028,
       116773, 120598, 114096, 102758, 106048, 115879, 113463, 100339,
       104339, 111938, 120285, 114645, 109933, 111049,  84875,  89820,
        85621,  96068,  80898,  83812,  82240,  92859,  81700,  86257,
        83222, 156391,  75212,  67725,  76277,  73194,  70460,  76774,
        67486,  60815,  64571,  54263,  50513])

## VERIFICACIÓN de OTs en Mongo DB

1. Se extrae el listado de todos los `id_ot` de los PDF existentes
2. Se verifica este listado con los documentos en `MongoDB`
3. Se verifica este listado con los documentos en `DeltaLake`

### Conexion con DASK

In [18]:
# 1. Listado de OTs con sus ID:

# Directorio Raiz de las OT (año) para validar

#root_dir = ("/home/vlad/OneDrive/01 JEZO/01 ACTIVIDADES DIARIAS DE TRABAJO DE LAS AGENCIAS/"
#            +
#            "2020")

root_dir = "/home/vlad/Documents/000 OTs Antiguas/2020"

list_pdfs = []
for path in Path( root_dir ).glob("**/*.pdf"):
    list_pdfs.append( str(path) )
    list_pdfs.sort()

print(f" Se han encontrado un total de: {len(list_pdfs)} Ordenes de Trabajo" )

start_time = time.time()
start_datetime = datetime.now()
print( f"Hora de inicio: {start_datetime.strftime('%Y-%m-%d %H:%M:%S')}\n\n" )

# Helper function to call the method on the result of a future
def call_load_ot(orden_trabajo_object):
    """
    Takes the result of the first task (an OrdenTrabajo object) 
    and calls the load_ot() method on it.
    """
    return orden_trabajo_object.load_ot()

# 1. Submit the first batch of tasks
# This returns a list of futures, same as before.
futures_step1 = [dask.submit(OrdenTrabajo.GestionOt, file) for file in list_pdfs]

# 2. Submit the second batch of tasks, feeding the first futures as input
futures_step2 = [dask.submit(call_load_ot, f) for f in futures_step1]

# 3. Now, gather only the FINAL results
# This single call executes the entire graph (both GestionOt and load_ot) in parallel.
obj_lists_dask = dask.gather(futures_step2)


end_time = time.time()
elapsed_time = end_time - start_time
end_datetime = datetime.now()
print(f"\n\n   Procesados todos los {len(obj_lists_dask)} items. Tiempo transcurrido: {elapsed_time:.2f} segundos.\n   Hora Final : {end_datetime.strftime('%Y-%m-%d %H:%M:%S')}")




 Se han encontrado un total de: 3054 Ordenes de Trabajo
Hora de inicio: 2025-07-24 10:21:50




   Procesados todos los 3054 items. Tiempo transcurrido: 294.74 segundos.
   Hora Final : 2025-07-24 10:26:45
